# 🖥️ Tmux Mastery Guide

**From beginner to productive power user**

*Self-study notebook — built with the Learning Notebook Builder*

---

**Version History**

| Date | Change |
|------|--------|
| 2026-04-21 | Part 4: Appendix & Cheat Sheet completed |
| 2026-04-21 | Part 3: Hands-On Lab completed (25 cells, 8 sub-files) |
| 2026-04-21 | Part 2: Core Concepts & Workflows completed (29 cells, 9 sub-files) |
| 2026-04-21 | Part 1: Background & Fundamentals completed (37 cells, 11 sub-files) |
| 2026-04-21 | Initial scaffold |

---

## 📋 Table of Contents

| Part | Topic | Status | Study Time |
|------|-------|--------|------------|
| 1 | Background & Fundamentals | ✅ Ready | ~2 hours |
| 2 | Core Concepts & Workflows | ✅ Ready | ~2 hours |
| 3 | Hands-On Lab | ✅ Ready | ~2 hours |
| 4 | Appendix & Cheat Sheet | ✅ Ready | ~1 hour |

**Total estimated study time: ~7 hours**

---

---

# Part 1: Background & Fundamentals

> **Target study time: ~2 hours** · Read, understand, and absorb — not just skim.

---

## 1.1 What Is a Terminal Multiplexer?

Before diving into tmux, let's understand the problem it solves.

When you open a terminal on your Mac or SSH into a remote server, you get **one shell**. One prompt. One running program at a time. If you want to run a second program, you need a second terminal window. If your SSH connection drops, everything you were running is gone.

A **terminal multiplexer** is a program that runs inside your terminal and lets you:

- Run **multiple terminal sessions** inside a single window
- **Split** your screen into panes (side-by-side or stacked)
- **Detach** from your sessions and **reattach** later — even from a different computer
- Keep programs running in the background after you disconnect

Think of it as a **window manager for your terminal**. Just like macOS lets you have multiple app windows on your desktop, a terminal multiplexer lets you have multiple shells inside one terminal.

### 🤔 Think About It

You SSH into a remote server and start a long-running database migration. Your laptop goes to sleep, killing the SSH connection. What happens to the migration?

**Without tmux:** The migration process receives a SIGHUP signal and terminates. You have to start over, potentially leaving the database in a partially migrated state.

**With tmux:** The migration keeps running inside the tmux session on the server. You reconnect, reattach to tmux, and see exactly where it is. Nothing was lost.

This is the core value proposition of tmux — **session persistence**.

## 1.2 The History of tmux

tmux stands for **terminal multiplexer**. It was created by **Nicholas Marriott** and first released on **20 November 2007**. It was developed as part of the OpenBSD project and first appeared in OpenBSD 4.6.

tmux was designed as a modern, cleaner replacement for **GNU Screen**, which had been the dominant terminal multiplexer since the 1980s. Screen had accumulated decades of code and was becoming difficult to maintain and extend.

Key milestones:

| Year | Event |
|------|-------|
| 1987 | GNU Screen first released |
| 2007 | tmux first released by Nicholas Marriott |
| 2009 | tmux included in OpenBSD base system |
| 2011 | tmux becomes default in many Linux distributions |
| 2020s | tmux becomes the de facto standard terminal multiplexer |
| 2024-2026 | tmux sees renewed interest due to AI coding tools (Claude Code, Kiro CLI, etc.) |

tmux is written in C, is open source (ISC license), and is actively maintained on GitHub at [github.com/tmux/tmux](https://github.com/tmux/tmux). The latest releases come approximately every six months.

> **Source:** [tmux GitHub Wiki](https://github.com/tmux/tmux/wiki), [Wikipedia](https://en.wikipedia.org/wiki/Tmux)

## 1.3 tmux vs. GNU Screen vs. Zellij

There are three main terminal multiplexers in use today. Here's how they compare:

| Feature | **tmux** | **GNU Screen** | **Zellij** |
|---------|----------|----------------|------------|
| First release | 2007 | 1987 | 2021 |
| Language | C | C | Rust |
| Configuration | `.tmux.conf` (custom syntax) | `.screenrc` | YAML/KDL |
| Plugin system | Via TPM (community) | Limited | Built-in (WebAssembly) |
| Pane splits | ✅ Horizontal & vertical | ✅ (added later) | ✅ Built-in |
| Mouse support | ✅ Configurable | Limited | ✅ Out of the box |
| Scriptability | ✅ Excellent | ✅ Good | ✅ Good |
| Learning curve | Medium | Medium | Low (shows keybindings on screen) |
| Beginner-friendly | ❌ Needs configuration | ❌ Dated interface | ✅ Works out of the box |
| Ecosystem & community | ✅ Massive | Declining | Growing |
| AI tool integration | ✅ Widely adopted | Rare | Emerging |
| Actively maintained | ✅ Yes | Minimal | ✅ Yes |

### Why tmux wins for most users

- **Ecosystem**: Thousands of blog posts, configs, plugins, and integrations. When you Google a problem, you'll find a tmux answer.
- **Scriptability**: tmux's command system is extremely powerful for automation — critical for AI tool workflows.
- **Stability**: Nearly two decades of battle-testing. It just works.
- **AI tool adoption**: Claude Code, Kiro CLI, and other AI coding tools have standardized on tmux for their workflows.

### When to consider Zellij instead

- If you want something that works beautifully out of the box with zero configuration
- If you prefer a mode-based interface with on-screen keybinding hints
- If you don't need deep scripting or AI tool integration (yet)

> **Bottom line:** For this guide, we're learning tmux because it's the industry standard and the tool that AI coding workflows have converged on. The skills transfer well if you later try Zellij.

### 🤔 Common Misconception

**"tmux is hard to learn"**

This is the #1 reason people avoid tmux. The truth: tmux's *defaults* are unintuitive (who decided `Ctrl+b %` should split a window?), but the *concepts* are simple. With a good `.tmux.conf` file, tmux becomes very natural. We'll build one together in Part 2.

The real learning curve is about 30 minutes of practice. After that, it's muscle memory.

## 1.4 The tmux Architecture: Server, Client, Sessions, Windows, Panes

This is the most important section in Part 1. Understanding tmux's architecture makes everything else click.

tmux uses a **client-server model**:

- The **tmux server** is a background process that manages all your sessions, windows, and panes. It keeps everything alive even when you disconnect.
- A **tmux client** is your terminal that connects to the server. When you run `tmux`, your terminal becomes a client.

The server starts automatically when you first run tmux and exits when there are no more sessions.

![Figure 0](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20320%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%20%20%3Cdefs%3E%3Cmarker%20id%3D%22arrowhead%22%20markerWidth%3D%2210%22%20markerHeight%3D%227%22%20refX%3D%2210%22%20refY%3D%223.5%22%20orient%3D%22auto%22%3E%3Cpolygon%20points%3D%220%200%2C%2010%203.5%2C%200%207%22%20fill%3D%22%23687078%22%2F%3E%3C%2Fmarker%3E%3C%2Fdefs%3E%0A%0A%20%20%3C%21--%20Server%20zone%20--%3E%0A%20%20%3Crect%20x%3D%2220%22%20y%3D%2210%22%20width%3D%22760%22%20height%3D%22300%22%20rx%3D%2212%22%20fill%3D%22%23F2F3F3%22%20stroke%3D%22%23D5DBDB%22%20stroke-width%3D%221%22%20stroke-dasharray%3D%226%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%2232%22%20y%3D%2230%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3Etmux%20server%20%28background%20process%29%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Session%201%20--%3E%0A%20%20%3Crect%20x%3D%2240%22%20y%3D%2245%22%20width%3D%22350%22%20height%3D%22245%22%20rx%3D%2212%22%20fill%3D%22%23FFFFFF%22%20stroke%3D%22%23D5DBDB%22%20stroke-width%3D%221%22%20stroke-dasharray%3D%226%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%2252%22%20y%3D%2265%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3ESession%3A%20%27work%27%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%2260%22%20y%3D%2280%22%20width%3D%22160%22%20height%3D%22100%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%23527FFF%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%2260%22%20y%3D%2280%22%20width%3D%22160%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%23527FFF%22%2F%3E%0A%20%20%3Crect%20x%3D%2260%22%20y%3D%2296%22%20width%3D%22160%22%20height%3D%2216%22%20fill%3D%22%23527FFF%22%2F%3E%0A%20%20%3Ctext%20x%3D%22140%22%20y%3D%22102%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EWindow%200%3A%20code%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22130%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EPane%200%3A%20vim%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22148%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EPane%201%3A%20shell%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22230%22%20y%3D%2280%22%20width%3D%22160%22%20height%3D%22100%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%23527FFF%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22230%22%20y%3D%2280%22%20width%3D%22160%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%23527FFF%22%2F%3E%0A%20%20%3Crect%20x%3D%22230%22%20y%3D%2296%22%20width%3D%22160%22%20height%3D%2216%22%20fill%3D%22%23527FFF%22%2F%3E%0A%20%20%3Ctext%20x%3D%22310%22%20y%3D%22102%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EWindow%201%3A%20logs%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22240%22%20y%3D%22130%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EPane%200%3A%20tail%20-f%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Session%202%20--%3E%0A%20%20%3Crect%20x%3D%22410%22%20y%3D%2245%22%20width%3D%22350%22%20height%3D%22245%22%20rx%3D%2212%22%20fill%3D%22%23FFFFFF%22%20stroke%3D%22%23D5DBDB%22%20stroke-width%3D%221%22%20stroke-dasharray%3D%226%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%22422%22%20y%3D%2265%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3ESession%3A%20%27personal%27%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22430%22%20y%3D%2280%22%20width%3D%22160%22%20height%3D%22100%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%238C4FFF%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22430%22%20y%3D%2280%22%20width%3D%22160%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%238C4FFF%22%2F%3E%0A%20%20%3Crect%20x%3D%22430%22%20y%3D%2296%22%20width%3D%22160%22%20height%3D%2216%22%20fill%3D%22%238C4FFF%22%2F%3E%0A%20%20%3Ctext%20x%3D%22510%22%20y%3D%22102%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EWindow%200%3A%20notes%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22440%22%20y%3D%22130%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EPane%200%3A%20vim%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22600%22%20y%3D%2280%22%20width%3D%22160%22%20height%3D%22100%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%238C4FFF%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22600%22%20y%3D%2280%22%20width%3D%22160%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%238C4FFF%22%2F%3E%0A%20%20%3Crect%20x%3D%22600%22%20y%3D%2296%22%20width%3D%22160%22%20height%3D%2216%22%20fill%3D%22%238C4FFF%22%2F%3E%0A%20%20%3Ctext%20x%3D%22680%22%20y%3D%22102%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EWindow%201%3A%20music%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22610%22%20y%3D%22130%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EPane%200%3A%20cmus%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Clients%20--%3E%0A%20%20%3Crect%20x%3D%22100%22%20y%3D%22230%22%20width%3D%22120%22%20height%3D%2240%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22160%22%20y%3D%22255%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2214%22%20font-weight%3D%22600%22%3EClient%201%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22540%22%20y%3D%22230%22%20width%3D%22120%22%20height%3D%2240%22%20rx%3D%2210%22%20fill%3D%22%23FF9900%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22600%22%20y%3D%22255%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2214%22%20font-weight%3D%22600%22%3EClient%202%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Arrows%20--%3E%0A%20%20%3Cline%20x1%3D%22160%22%20y1%3D%22230%22%20x2%3D%22160%22%20y2%3D%22175%22%20stroke%3D%22%232EA043%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Ctext%20x%3D%22160%22%20y%3D%22194%22%20text-anchor%3D%22middle%22%20fill%3D%22%232EA043%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Eattached%3C%2Ftext%3E%0A%20%20%3Cline%20x1%3D%22600%22%20y1%3D%22230%22%20x2%3D%22600%22%20y2%3D%22175%22%20stroke%3D%22%23FF9900%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Ctext%20x%3D%22600%22%20y%3D%22194%22%20text-anchor%3D%22middle%22%20fill%3D%22%23FF9900%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Eattached%3C%2Ftext%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%22310%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EMultiple%20clients%20can%20attach%20to%20different%20sessions%20on%20the%20same%20server%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 1: tmux client-server architecture with sessions, windows, and panes*

### The Hierarchy: Server → Sessions → Windows → Panes

tmux organizes everything in a strict hierarchy:

```
tmux server
├── Session: "work"
│   ├── Window 0: "code"     ← current window
│   │   ├── Pane 0: vim      ← active pane
│   │   └── Pane 1: shell
│   └── Window 1: "logs"
│       └── Pane 0: tail -f
└── Session: "personal"
    ├── Window 0: "notes"
    │   └── Pane 0: vim
    └── Window 1: "music"
        └── Pane 0: cmus
```

Let's define each level:

### Sessions

A **session** is a collection of windows. Think of it as a **workspace** or **project context**.

- Each session has a **name** (e.g., "work", "personal", "customer-project")
- A session can be **attached** (visible on your screen) or **detached** (running in the background)
- Multiple clients can attach to the same session simultaneously (useful for pair programming)
- You typically create one session per project or context

**Key insight:** When you "detach" from tmux, you're detaching the *client* from the *session*. The session (and everything running in it) stays alive on the server.

### Windows

A **window** is like a **tab** in your browser. Each session contains one or more windows.

- Each window has a **name** (defaults to the running program, e.g., "bash", "vim")
- Each window has an **index** (number) starting from 0
- One window in each session is the **current window** — the one you see
- The previous current window is called the **last window**
- Windows appear in the **status bar** at the bottom of the screen

**Analogy:** If a session is a browser, windows are tabs. You can have many open, but you see one at a time.

### Panes

A **pane** is a **split** within a window. Each pane contains one terminal running one program.

- A window can be split horizontally (top/bottom) or vertically (left/right)
- Each pane runs its own shell or program independently
- One pane in each window is the **active pane** — where your keystrokes go
- The active pane's border is highlighted in green
- Panes can be resized, swapped, and zoomed (temporarily made full-screen)

**Analogy:** If a window is a browser tab, panes are like having multiple browser frames within that tab.

![Figure 1](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20400%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%20%20%3Cdefs%3E%3Cmarker%20id%3D%22arrowhead%22%20markerWidth%3D%2210%22%20markerHeight%3D%227%22%20refX%3D%2210%22%20refY%3D%223.5%22%20orient%3D%22auto%22%3E%3Cpolygon%20points%3D%220%200%2C%2010%203.5%2C%200%207%22%20fill%3D%22%23687078%22%2F%3E%3C%2Fmarker%3E%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2225%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2216%22%20font-weight%3D%22700%22%3Etmux%20Hierarchy%3A%20Server%20%E2%86%92%20Session%20%E2%86%92%20Window%20%E2%86%92%20Pane%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Server%20--%3E%0A%20%20%3Crect%20x%3D%22320%22%20y%3D%2245%22%20width%3D%22160%22%20height%3D%2240%22%20rx%3D%2210%22%20fill%3D%22%23232F3E%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2270%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2214%22%20font-weight%3D%22600%22%3Etmux%20server%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Sessions%20--%3E%0A%20%20%3Cline%20x1%3D%22360%22%20y1%3D%2285%22%20x2%3D%22200%22%20y2%3D%22130%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Cline%20x1%3D%22440%22%20y1%3D%2285%22%20x2%3D%22600%22%20y2%3D%22130%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22120%22%20y%3D%22130%22%20width%3D%22160%22%20height%3D%2240%22%20rx%3D%2210%22%20fill%3D%22%23527FFF%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22200%22%20y%3D%22155%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2214%22%20font-weight%3D%22600%22%3ESession%3A%20work%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22520%22%20y%3D%22130%22%20width%3D%22160%22%20height%3D%2240%22%20rx%3D%2210%22%20fill%3D%22%238C4FFF%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22600%22%20y%3D%22155%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2214%22%20font-weight%3D%22600%22%3ESession%3A%20personal%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Windows%20under%20work%20--%3E%0A%20%20%3Cline%20x1%3D%22160%22%20y1%3D%22170%22%20x2%3D%22100%22%20y2%3D%22215%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Cline%20x1%3D%22240%22%20y1%3D%22170%22%20x2%3D%22280%22%20y2%3D%22215%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Crect%20x%3D%2230%22%20y%3D%22215%22%20width%3D%22140%22%20height%3D%2236%22%20rx%3D%2210%22%20fill%3D%22%2300A1C9%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22100%22%20y%3D%22238%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2214%22%20font-weight%3D%22600%22%3EWindow%200%3A%20code%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22210%22%20y%3D%22215%22%20width%3D%22140%22%20height%3D%2236%22%20rx%3D%2210%22%20fill%3D%22%2300A1C9%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22280%22%20y%3D%22238%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2214%22%20font-weight%3D%22600%22%3EWindow%201%3A%20logs%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Windows%20under%20personal%20--%3E%0A%20%20%3Cline%20x1%3D%22560%22%20y1%3D%22170%22%20x2%3D%22500%22%20y2%3D%22215%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Cline%20x1%3D%22640%22%20y1%3D%22170%22%20x2%3D%22680%22%20y2%3D%22215%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22430%22%20y%3D%22215%22%20width%3D%22140%22%20height%3D%2236%22%20rx%3D%2210%22%20fill%3D%22%2300A1C9%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22500%22%20y%3D%22238%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2214%22%20font-weight%3D%22600%22%3EWindow%200%3A%20notes%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22610%22%20y%3D%22215%22%20width%3D%22140%22%20height%3D%2236%22%20rx%3D%2210%22%20fill%3D%22%2300A1C9%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22680%22%20y%3D%22238%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2214%22%20font-weight%3D%22600%22%3EWindow%201%3A%20music%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Panes%20under%20Window%200%3A%20code%20--%3E%0A%20%20%3Cline%20x1%3D%2270%22%20y1%3D%22251%22%20x2%3D%2250%22%20y2%3D%22295%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Cline%20x1%3D%22130%22%20y1%3D%22251%22%20x2%3D%22150%22%20y2%3D%22295%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Crect%20x%3D%2210%22%20y%3D%22295%22%20width%3D%2280%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%2250%22%20y%3D%22316%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3EPane%3A%20vim%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22110%22%20y%3D%22295%22%20width%3D%2280%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22150%22%20y%3D%22316%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3EPane%3A%20shell%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Pane%20under%20Window%201%3A%20logs%20--%3E%0A%20%20%3Cline%20x1%3D%22280%22%20y1%3D%22251%22%20x2%3D%22280%22%20y2%3D%22295%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22240%22%20y%3D%22295%22%20width%3D%2280%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22280%22%20y%3D%22316%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3EPane%3A%20tail%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Pane%20under%20notes%20--%3E%0A%20%20%3Cline%20x1%3D%22500%22%20y1%3D%22251%22%20x2%3D%22500%22%20y2%3D%22295%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22460%22%20y%3D%22295%22%20width%3D%2280%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22500%22%20y%3D%22316%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3EPane%3A%20vim%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Pane%20under%20music%20--%3E%0A%20%20%3Cline%20x1%3D%22680%22%20y1%3D%22251%22%20x2%3D%22680%22%20y2%3D%22295%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22640%22%20y%3D%22295%22%20width%3D%2280%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22680%22%20y%3D%22316%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3EPane%3A%20cmus%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Legend%20--%3E%0A%20%20%3Crect%20x%3D%22200%22%20y%3D%22355%22%20width%3D%2280%22%20height%3D%2224%22%20rx%3D%2210%22%20fill%3D%22%23232F3E%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22240%22%20y%3D%22372%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3EServer%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22300%22%20y%3D%22355%22%20width%3D%2280%22%20height%3D%2224%22%20rx%3D%2210%22%20fill%3D%22%23527FFF%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22340%22%20y%3D%22372%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3ESession%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22400%22%20y%3D%22355%22%20width%3D%2280%22%20height%3D%2224%22%20rx%3D%2210%22%20fill%3D%22%2300A1C9%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22440%22%20y%3D%22372%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3EWindow%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22500%22%20y%3D%22355%22%20width%3D%2280%22%20height%3D%2224%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22540%22%20y%3D%22372%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3EPane%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 2: The complete tmux hierarchy — each level contains the next*

### Summary of Terms

| Term | Description | Analogy |
|------|-------------|---------|
| **Server** | Background process managing all state | The operating system |
| **Client** | Your terminal attached to a session | A monitor/display |
| **Session** | A workspace grouping windows | A virtual desktop |
| **Window** | A tab within a session | A browser tab |
| **Pane** | A split within a window running one program | A frame within a tab |
| **Active pane** | The pane receiving keystrokes (green border) | The focused text field |
| **Current window** | The visible window in the attached session | The active browser tab |
| **Prefix key** | The key combo that tells tmux "the next key is for you" | A modifier key |

### 🤔 Think About It

Before reading on, predict: if you have 2 sessions, each with 3 windows, and each window has 2 panes — how many terminals are running inside tmux?

**Answer:** 2 × 3 × 2 = **12 terminals**, all managed by a single tmux server, accessible from a single terminal window on your screen.

## 1.5 The Prefix Key — How You Talk to tmux

This is the concept that trips up every beginner. Let's get it right.

When you're inside tmux, every key you press goes to the program running in the active pane (usually your shell). So how does tmux know when a key is meant for *it* rather than for the shell?

The answer is the **prefix key**. By default, this is **`Ctrl+b`**.

Here's how it works:

1. You press **`Ctrl+b`** (the prefix) — nothing visible happens, but tmux is now listening
2. You release the keys
3. You press the **command key** — tmux executes the corresponding command

For example, to create a new window:
- Press `Ctrl+b`, release, then press `c`
- This is written as **`C-b c`** in tmux documentation

> ⚠️ **Critical detail:** You must **release** `Ctrl+b` before pressing the command key. `Ctrl+b` then `c` is different from `Ctrl+b+c` (holding all three).

![Figure 2](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20750%20200%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%20%20%3Cdefs%3E%3Cmarker%20id%3D%22arrowhead%22%20markerWidth%3D%2210%22%20markerHeight%3D%227%22%20refX%3D%2210%22%20refY%3D%223.5%22%20orient%3D%22auto%22%3E%3Cpolygon%20points%3D%220%200%2C%2010%203.5%2C%200%207%22%20fill%3D%22%23687078%22%2F%3E%3C%2Fmarker%3E%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22375%22%20y%3D%2225%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2215%22%20font-weight%3D%22700%22%3EHow%20the%20Prefix%20Key%20Works%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Step%201%20--%3E%0A%20%20%3Crect%20x%3D%2230%22%20y%3D%2255%22%20width%3D%22140%22%20height%3D%2250%22%20rx%3D%2210%22%20fill%3D%22%23527FFF%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22100%22%20y%3D%2285%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22600%22%3E%E2%91%A0%20Press%20Ctrl%2Bb%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22100%22%20y%3D%22130%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Etmux%20enters%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22100%22%20y%3D%22143%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%22command%20mode%22%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22170%22%20y1%3D%2280%22%20x2%3D%22230%22%20y2%3D%2280%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%0A%20%20%3C%21--%20Step%202%20--%3E%0A%20%20%3Crect%20x%3D%22230%22%20y%3D%2255%22%20width%3D%22140%22%20height%3D%2250%22%20rx%3D%2210%22%20fill%3D%22%23FF9900%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22300%22%20y%3D%2285%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22600%22%3E%E2%91%A1%20Release%20keys%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22300%22%20y%3D%22130%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Etmux%20waits%20for%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22300%22%20y%3D%22143%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Enext%20keypress%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22370%22%20y1%3D%2280%22%20x2%3D%22430%22%20y2%3D%2280%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%0A%20%20%3C%21--%20Step%203%20--%3E%0A%20%20%3Crect%20x%3D%22430%22%20y%3D%2255%22%20width%3D%22140%22%20height%3D%2250%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22500%22%20y%3D%2285%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22600%22%3E%E2%91%A2%20Press%20command%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22500%22%20y%3D%22130%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Ee.g.%2C%20%27c%27%20%3D%20new%20window%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22500%22%20y%3D%22143%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%27d%27%20%3D%20detach%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22570%22%20y1%3D%2280%22%20x2%3D%22620%22%20y2%3D%2280%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%0A%20%20%3C%21--%20Result%20--%3E%0A%20%20%3Crect%20x%3D%22620%22%20y%3D%2255%22%20width%3D%22110%22%20height%3D%2250%22%20rx%3D%2210%22%20fill%3D%22%238C4FFF%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22675%22%20y%3D%2285%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22600%22%3E%E2%9C%85%20Action%21%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22675%22%20y%3D%22130%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Etmux%20executes%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22675%22%20y%3D%22143%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Ethe%20command%3C%2Ftext%3E%0A%0A%20%20%3Ctext%20x%3D%22375%22%20y%3D%22180%22%20text-anchor%3D%22middle%22%20fill%3D%22%23D13212%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EIf%20you%20press%20an%20unbound%20key%20after%20the%20prefix%2C%20nothing%20happens%20%E2%80%94%20tmux%20just%20ignores%20it%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 3: The three-step prefix key flow*

### The Most Important Default Key Bindings

You don't need to memorize all of these now — we'll practice them in Part 3. But here are the ones you'll use in the first 10 minutes:

| Keys | Action | What it does |
|------|--------|-------------|
| `C-b d` | **Detach** | Disconnect from tmux (session keeps running) |
| `C-b c` | **Create window** | Opens a new window with a shell |
| `C-b n` | **Next window** | Switch to the next window |
| `C-b p` | **Previous window** | Switch to the previous window |
| `C-b 0-9` | **Go to window N** | Jump to window by number |
| `C-b %` | **Split vertical** | Split pane left/right |
| `C-b "` | **Split horizontal** | Split pane top/bottom |
| `C-b ←↑↓→` | **Move between panes** | Switch active pane |
| `C-b z` | **Zoom pane** | Toggle pane full-screen |
| `C-b ?` | **Help** | Show all key bindings |

> **Notation:** `C-b` means `Ctrl+b`. `C-b c` means press `Ctrl+b`, release, then press `c`.

### Why `Ctrl+b`? (And Why Many People Change It)

The default prefix `Ctrl+b` was chosen because it doesn't conflict with common programs. But it's ergonomically awkward — your fingers have to stretch.

Many users remap the prefix to **`Ctrl+a`** (which was GNU Screen's default) or **`Ctrl+Space`**. We'll cover how to do this in Part 2 when we build a `.tmux.conf`.

For now, use the default `Ctrl+b` so you learn the standard first.

### 🤔 Mini-Exercise

Without looking at the table above, answer these:

1. How do you detach from tmux? → `C-b d`
2. How do you create a new window? → `C-b c`
3. How do you split a pane vertically (left/right)? → `C-b %`
4. How do you see all key bindings? → `C-b ?`

If you got 3 or more right, you're ready to move on. If not, re-read the table — these four commands are your survival kit.

## 1.6 The Detach/Attach Lifecycle — tmux's Killer Feature

You already know the basic idea: detach from tmux, and your session keeps running. Let's understand exactly what happens at each step, because this is what makes tmux indispensable.

![Figure 3](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20350%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%20%20%3Cdefs%3E%3Cmarker%20id%3D%22arrowhead%22%20markerWidth%3D%2210%22%20markerHeight%3D%227%22%20refX%3D%2210%22%20refY%3D%223.5%22%20orient%3D%22auto%22%3E%3Cpolygon%20points%3D%220%200%2C%2010%203.5%2C%200%207%22%20fill%3D%22%23687078%22%2F%3E%3C%2Fmarker%3E%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2222%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2215%22%20font-weight%3D%22700%22%3EThe%20Detach%20%2F%20Attach%20Lifecycle%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Timeline%20--%3E%0A%20%20%3Cline%20x1%3D%2250%22%20y1%3D%22170%22%20x2%3D%22750%22%20y2%3D%22170%22%20stroke%3D%22%23D5DBDB%22%20stroke-width%3D%222%22%2F%3E%0A%0A%20%20%3C%21--%20Step%201%3A%20Start%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%2250%22%20width%3D%22130%22%20height%3D%2245%22%20rx%3D%2210%22%20fill%3D%22%23527FFF%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22115%22%20y%3D%2277%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3Etmux%20new%20-s%20work%3C%2Ftext%3E%0A%20%20%3Cline%20x1%3D%22115%22%20y1%3D%2295%22%20x2%3D%22115%22%20y2%3D%22170%22%20stroke%3D%22%23527FFF%22%20stroke-width%3D%222%22%20stroke-dasharray%3D%224%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%22115%22%20y%3D%22200%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3EServer%20starts%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22115%22%20y%3D%22213%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3ESession%20created%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22115%22%20y%3D%22226%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3EClient%20attached%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Step%202%3A%20Work%20--%3E%0A%20%20%3Crect%20x%3D%22220%22%20y%3D%2250%22%20width%3D%22100%22%20height%3D%2245%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22270%22%20y%3D%2277%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EYou%20work...%3C%2Ftext%3E%0A%20%20%3Cline%20x1%3D%22270%22%20y1%3D%2295%22%20x2%3D%22270%22%20y2%3D%22170%22%20stroke%3D%22%232EA043%22%20stroke-width%3D%222%22%20stroke-dasharray%3D%224%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%22270%22%20y%3D%22200%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3EPrograms%20run%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22270%22%20y%3D%22213%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3Ein%20panes%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Step%203%3A%20Detach%20--%3E%0A%20%20%3Crect%20x%3D%22360%22%20y%3D%2250%22%20width%3D%22100%22%20height%3D%2245%22%20rx%3D%2210%22%20fill%3D%22%23FF9900%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22410%22%20y%3D%2277%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22600%22%3EC-b%20d%3C%2Ftext%3E%0A%20%20%3Cline%20x1%3D%22410%22%20y1%3D%2295%22%20x2%3D%22410%22%20y2%3D%22170%22%20stroke%3D%22%23FF9900%22%20stroke-width%3D%222%22%20stroke-dasharray%3D%224%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%22410%22%20y%3D%22200%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3EClient%20exits%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22410%22%20y%3D%22213%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3ESession%20stays%20alive%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22410%22%20y%3D%22226%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3EPrograms%20keep%20running%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Step%204%3A%20Time%20passes%20--%3E%0A%20%20%3Ctext%20x%3D%22510%22%20y%3D%2265%22%20text-anchor%3D%22middle%22%20fill%3D%22%23D13212%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22600%22%3E%E2%8F%B0%20Hours%20pass...%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22510%22%20y%3D%2282%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3ELaptop%20sleeps%2C%20SSH%20drops%2C%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22510%22%20y%3D%2295%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3Eyou%20go%20home%20%E2%80%94%20doesn%27t%20matter%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Step%205%3A%20Reattach%20--%3E%0A%20%20%3Crect%20x%3D%22610%22%20y%3D%2250%22%20width%3D%22130%22%20height%3D%2245%22%20rx%3D%2210%22%20fill%3D%22%238C4FFF%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22675%22%20y%3D%2277%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3Etmux%20attach%20-t%20work%3C%2Ftext%3E%0A%20%20%3Cline%20x1%3D%22675%22%20y1%3D%2295%22%20x2%3D%22675%22%20y2%3D%22170%22%20stroke%3D%22%238C4FFF%22%20stroke-width%3D%222%22%20stroke-dasharray%3D%224%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%22675%22%20y%3D%22200%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3ENew%20client%20attaches%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22675%22%20y%3D%22213%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3ESame%20session%2C%20same%20state%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22675%22%20y%3D%22226%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3EEven%20from%20different%20machine%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Server%20bar%20--%3E%0A%20%20%3Crect%20x%3D%2280%22%20y%3D%22270%22%20width%3D%22660%22%20height%3D%2230%22%20rx%3D%226%22%20fill%3D%22%23F2F3F3%22%20stroke%3D%22%23D5DBDB%22%20stroke-width%3D%221%22%2F%3E%0A%20%20%3Ctext%20x%3D%22410%22%20y%3D%22290%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3Etmux%20server%3A%20always%20running%20on%20the%20host%20machine%20%E2%9C%85%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Annotations%20--%3E%0A%20%20%3Crect%20x%3D%2280%22%20y%3D%22310%22%20width%3D%22250%22%20height%3D%2222%22%20rx%3D%224%22%20fill%3D%22%23FF9900%22%20opacity%3D%220.15%22%2F%3E%0A%20%20%3Ctext%20x%3D%22205%22%20y%3D%22325%22%20text-anchor%3D%22middle%22%20fill%3D%22%23FF9900%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3EClient%20disconnected%20%E2%80%94%20no%20problem%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22490%22%20y%3D%22310%22%20width%3D%22250%22%20height%3D%2222%22%20rx%3D%224%22%20fill%3D%22%238C4FFF%22%20opacity%3D%220.15%22%2F%3E%0A%20%20%3Ctext%20x%3D%22615%22%20y%3D%22325%22%20text-anchor%3D%22middle%22%20fill%3D%22%238C4FFF%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3EClient%20reconnected%20%E2%80%94%20everything%20intact%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 4: The detach/attach lifecycle — your session survives disconnections*

### The Essential Commands for Session Management

Here are the shell commands (run outside tmux) and key bindings (run inside tmux) for the attach/detach lifecycle:

**From the shell (outside tmux):**

| Command | What it does |
|---------|-------------|
| `tmux` | Start a new unnamed session and attach |
| `tmux new -s work` | Start a new session named "work" and attach |
| `tmux ls` | List all sessions |
| `tmux attach` or `tmux a` | Attach to the most recent session |
| `tmux attach -t work` | Attach to the session named "work" |
| `tmux new -As work` | Attach to "work" if it exists, or create it |
| `tmux kill-session -t work` | Kill the session named "work" |
| `tmux kill-server` | Kill the tmux server and all sessions |

**From inside tmux (with prefix key):**

| Keys | What it does |
|------|-------------|
| `C-b d` | Detach from the current session |
| `C-b s` | Show session picker (tree mode) |
| `C-b $` | Rename the current session |
| `C-b (` | Switch to the previous session |
| `C-b )` | Switch to the next session |

### The `tmux attach || tmux` Pattern

You mentioned you already use this:

```bash
tmux attach || tmux
```

This is a smart one-liner that means:
1. Try to attach to an existing session
2. If no session exists (the `attach` command fails), start a new one

An even better version uses the `-A` flag:

```bash
tmux new -As main
```

This does the same thing but with a named session called "main". If "main" exists, it attaches. If not, it creates it. One command, no `||` needed.

> **Pro tip:** Add `tmux new -As main` to your `.bashrc` or `.zshrc` on remote servers so you're always in tmux when you SSH in.

## 1.7 The Status Bar — Your tmux Dashboard

When you're inside tmux, there's a green bar at the bottom of your screen. This is the **status line** (or status bar), and it's your primary source of information about what's happening in tmux.

The default status bar shows:

```
[session-name] 0:bash  1:vim*  2:logs-                          "hostname" 14:30 21-Apr-26
├─ session     ├─ window list (current=*, last=-)                ├─ pane title  ├─ time/date
```

Let's break it down:

| Part | Meaning |
|------|---------|
| `[session-name]` | The name of the attached session (left side) |
| `0:bash` | Window index 0, named "bash" |
| `1:vim*` | Window index 1, named "vim" — the `*` means this is the **current window** |
| `2:logs-` | Window index 2, named "logs" — the `-` means this is the **last window** |
| `"hostname"` | The pane title (defaults to hostname) |
| `14:30 21-Apr-26` | Current time and date |

If you have more windows than fit on the screen, `<` or `>` arrows appear to indicate hidden windows.

![Figure 4](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20130%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%0A%20%20%3C%21--%20Status%20bar%20background%20--%3E%0A%20%20%3Crect%20x%3D%2220%22%20y%3D%2215%22%20width%3D%22760%22%20height%3D%2236%22%20rx%3D%224%22%20fill%3D%22%232EA043%22%2F%3E%0A%0A%20%20%3C%21--%20Session%20name%20--%3E%0A%20%20%3Ctext%20x%3D%2230%22%20y%3D%2238%22%20fill%3D%22white%22%20font-family%3D%22monospace%22%20font-size%3D%2213%22%20font-weight%3D%22600%22%3E%5Bwork%5D%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Windows%20--%3E%0A%20%20%3Ctext%20x%3D%2290%22%20y%3D%2238%22%20fill%3D%22%23C8E6C9%22%20font-family%3D%22monospace%22%20font-size%3D%2213%22%3E0%3Abash%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22155%22%20y%3D%2220%22%20width%3D%2260%22%20height%3D%2226%22%20rx%3D%223%22%20fill%3D%22white%22%20opacity%3D%220.2%22%2F%3E%0A%20%20%3Ctext%20x%3D%22162%22%20y%3D%2238%22%20fill%3D%22white%22%20font-family%3D%22monospace%22%20font-size%3D%2213%22%20font-weight%3D%22700%22%3E1%3Avim%2A%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22225%22%20y%3D%2238%22%20fill%3D%22%23C8E6C9%22%20font-family%3D%22monospace%22%20font-size%3D%2213%22%3E2%3Alogs-%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Right%20side%20--%3E%0A%20%20%3Ctext%20x%3D%22580%22%20y%3D%2238%22%20fill%3D%22%23C8E6C9%22%20font-family%3D%22monospace%22%20font-size%3D%2213%22%3E%22my-macbook%22%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22700%22%20y%3D%2238%22%20fill%3D%22white%22%20font-family%3D%22monospace%22%20font-size%3D%2213%22%3E14%3A30%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Annotations%20--%3E%0A%20%20%3Cline%20x1%3D%2255%22%20y1%3D%2255%22%20x2%3D%2255%22%20y2%3D%2275%22%20stroke%3D%22%23527FFF%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Ctext%20x%3D%2255%22%20y%3D%2290%22%20text-anchor%3D%22middle%22%20fill%3D%22%23527FFF%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3ESession%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22120%22%20y1%3D%2255%22%20x2%3D%22120%22%20y2%3D%2275%22%20stroke%3D%22%2300A1C9%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Ctext%20x%3D%22120%22%20y%3D%2290%22%20text-anchor%3D%22middle%22%20fill%3D%22%2300A1C9%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3EWindow%200%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22185%22%20y1%3D%2255%22%20x2%3D%22185%22%20y2%3D%2275%22%20stroke%3D%22%23FF9900%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Ctext%20x%3D%22185%22%20y%3D%2290%22%20text-anchor%3D%22middle%22%20fill%3D%22%23FF9900%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3ECurrent%20%28%2A%29%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22255%22%20y1%3D%2255%22%20x2%3D%22255%22%20y2%3D%2275%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Ctext%20x%3D%22255%22%20y%3D%2290%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3ELast%20%28-%29%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22630%22%20y1%3D%2255%22%20x2%3D%22630%22%20y2%3D%2275%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Ctext%20x%3D%22630%22%20y%3D%2290%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3EPane%20title%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22720%22%20y1%3D%2255%22%20x2%3D%22720%22%20y2%3D%2275%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%20%20%3Ctext%20x%3D%22720%22%20y%3D%2290%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3ETime%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 5: Anatomy of the tmux status bar*

### What the Status Bar Tells You at a Glance

The status bar is surprisingly information-dense. With practice, you'll glance at it and instantly know:

- **Which session** you're in (are you in "work" or "personal"?)
- **How many windows** are open and their names
- **Which window** is active (the `*` marker)
- **Which window** you were just in (the `-` marker — useful for `C-b l` to toggle back)
- **The time** (surprisingly useful when you're deep in a terminal session)

The status bar is fully customizable — colors, content, position (top or bottom), even dynamic content from shell commands. We'll customize it in Part 2.

## 1.8 Why tmux Matters for AI Coding Tools

This is the section that probably brought you here. AI coding tools like **Claude Code**, **Kiro CLI**, **Codex CLI**, and **Gemini CLI** are all terminal-based. They run long-running tasks, generate lots of output, and benefit enormously from tmux. Here's why:

![Figure 5](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20380%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%20%20%3Cdefs%3E%3Cmarker%20id%3D%22arrowhead%22%20markerWidth%3D%2210%22%20markerHeight%3D%227%22%20refX%3D%2210%22%20refY%3D%223.5%22%20orient%3D%22auto%22%3E%3Cpolygon%20points%3D%220%200%2C%2010%203.5%2C%200%207%22%20fill%3D%22%23687078%22%2F%3E%3C%2Fmarker%3E%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2222%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2215%22%20font-weight%3D%22700%22%3Etmux%20%2B%20AI%20Coding%20Tools%3A%20The%20Perfect%20Match%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Without%20tmux%20--%3E%0A%20%20%3Crect%20x%3D%2220%22%20y%3D%2240%22%20width%3D%22370%22%20height%3D%22320%22%20rx%3D%2212%22%20fill%3D%22%23FFFFFF%22%20stroke%3D%22%23D5DBDB%22%20stroke-width%3D%221%22%20stroke-dasharray%3D%226%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%2232%22%20y%3D%2260%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3E%E2%9D%8C%20Without%20tmux%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%2240%22%20y%3D%2275%22%20width%3D%22170%22%20height%3D%2290%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%23D13212%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%2240%22%20y%3D%2275%22%20width%3D%22170%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%23D13212%22%2F%3E%0A%20%20%3Crect%20x%3D%2240%22%20y%3D%2291%22%20width%3D%22170%22%20height%3D%2216%22%20fill%3D%22%23D13212%22%2F%3E%0A%20%20%3Ctext%20x%3D%22125%22%20y%3D%2297%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3ETerminal%201%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2250%22%20y%3D%22125%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EClaude%20Code%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2250%22%20y%3D%22143%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Eworking%20on%20feat%20A%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22220%22%20y%3D%2275%22%20width%3D%22170%22%20height%3D%2290%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%23D13212%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22220%22%20y%3D%2275%22%20width%3D%22170%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%23D13212%22%2F%3E%0A%20%20%3Crect%20x%3D%22220%22%20y%3D%2291%22%20width%3D%22170%22%20height%3D%2216%22%20fill%3D%22%23D13212%22%2F%3E%0A%20%20%3Ctext%20x%3D%22305%22%20y%3D%2297%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3ETerminal%202%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22230%22%20y%3D%22125%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EKiro%20CLI%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22230%22%20y%3D%22143%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Eworking%20on%20feat%20B%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22205%22%20y%3D%22185%22%20text-anchor%3D%22middle%22%20fill%3D%22%23D13212%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3E%E2%9A%A0%EF%B8%8F%20SSH%20drops%20%E2%86%92%20both%20sessions%20LOST%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22205%22%20y%3D%22200%22%20text-anchor%3D%22middle%22%20fill%3D%22%23D13212%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%E2%9A%A0%EF%B8%8F%20Can%27t%20see%20both%20at%20once%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22205%22%20y%3D%22215%22%20text-anchor%3D%22middle%22%20fill%3D%22%23D13212%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%E2%9A%A0%EF%B8%8F%20No%20scrollback%20for%20long%20output%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22205%22%20y%3D%22230%22%20text-anchor%3D%22middle%22%20fill%3D%22%23D13212%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%E2%9A%A0%EF%B8%8F%20Can%27t%20fire-and-forget%20tasks%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22205%22%20y%3D%22245%22%20text-anchor%3D%22middle%22%20fill%3D%22%23D13212%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%E2%9A%A0%EF%B8%8F%20One%20device%20only%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20With%20tmux%20--%3E%0A%20%20%3Crect%20x%3D%22410%22%20y%3D%2240%22%20width%3D%22370%22%20height%3D%22320%22%20rx%3D%2212%22%20fill%3D%22%23FFFFFF%22%20stroke%3D%22%23D5DBDB%22%20stroke-width%3D%221%22%20stroke-dasharray%3D%226%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%22422%22%20y%3D%2260%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3E%E2%9C%85%20With%20tmux%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22430%22%20y%3D%2275%22%20width%3D%22170%22%20height%3D%2290%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%232EA043%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22430%22%20y%3D%2275%22%20width%3D%22170%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%2F%3E%0A%20%20%3Crect%20x%3D%22430%22%20y%3D%2291%22%20width%3D%22170%22%20height%3D%2216%22%20fill%3D%22%232EA043%22%2F%3E%0A%20%20%3Ctext%20x%3D%22515%22%20y%3D%2297%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EWindow%201%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22440%22%20y%3D%22125%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EClaude%20Code%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22440%22%20y%3D%22143%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Eworking%20on%20feat%20A%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22610%22%20y%3D%2275%22%20width%3D%22170%22%20height%3D%2290%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%232EA043%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22610%22%20y%3D%2275%22%20width%3D%22170%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%2F%3E%0A%20%20%3Crect%20x%3D%22610%22%20y%3D%2291%22%20width%3D%22170%22%20height%3D%2216%22%20fill%3D%22%232EA043%22%2F%3E%0A%20%20%3Ctext%20x%3D%22695%22%20y%3D%2297%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EWindow%202%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22620%22%20y%3D%22125%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EKiro%20CLI%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22620%22%20y%3D%22143%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Eworking%20on%20feat%20B%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22595%22%20y%3D%22185%22%20text-anchor%3D%22middle%22%20fill%3D%22%232EA043%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3E%E2%9C%85%20SSH%20drops%20%E2%86%92%20sessions%20survive%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22595%22%20y%3D%22200%22%20text-anchor%3D%22middle%22%20fill%3D%22%232EA043%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%E2%9C%85%20Split%20panes%20to%20monitor%20both%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22595%22%20y%3D%22215%22%20text-anchor%3D%22middle%22%20fill%3D%22%232EA043%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%E2%9C%85%2050K%2B%20lines%20of%20scrollback%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22595%22%20y%3D%22230%22%20text-anchor%3D%22middle%22%20fill%3D%22%232EA043%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%E2%9C%85%20Detach%20and%20check%20back%20later%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22595%22%20y%3D%22245%22%20text-anchor%3D%22middle%22%20fill%3D%22%232EA043%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%E2%9C%85%20Reattach%20from%20any%20device%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Bottom%20summary%20--%3E%0A%20%20%3Crect%20x%3D%22200%22%20y%3D%22330%22%20width%3D%22400%22%20height%3D%2230%22%20rx%3D%2210%22%20fill%3D%22%23232F3E%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%22350%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3Etmux%20%3D%20session%20persistence%20%2B%20multiplexing%20%2B%20scrollback%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 6: AI coding tools with and without tmux*

### The Five Reasons AI Tools Need tmux

**1. Session Persistence (Fire and Forget)**

AI coding tools often run long tasks — refactoring a codebase, running test suites in a loop, generating code across multiple files. With tmux, you can:
- Start a task: "refactor all API handlers to use the new error type"
- Detach (`C-b d`)
- Close your laptop, go get coffee
- Come back, reattach (`tmux attach`), and see the results

This is the "fire and forget" pattern. Without tmux, closing your terminal kills the AI agent mid-task.

**2. Parallel AI Sessions**

With tmux windows and panes, you can run multiple AI agents simultaneously:
- Window 1: Claude Code working on feature A
- Window 2: Kiro CLI reviewing code
- Window 3: Your shell for manual testing

Switch between them with `C-b 1`, `C-b 2`, `C-b 3`. Or split panes to see two at once.

**3. Scrollback and Search**

AI tools produce a LOT of output. Your terminal's default scrollback buffer is usually 1,000-10,000 lines. tmux's `history-limit` can be set to 50,000+ lines. And tmux's copy mode (`C-b [`) lets you scroll back, search with `Ctrl+r`, and copy text — all without a mouse.

**4. Multi-Device Access**

Start an AI task on your work laptop. SSH from your phone to check on it. Reattach from your home computer to review the results. The session is on the server — any device can connect.

**5. Reproducible Layouts**

tmux can be scripted to create specific window/pane layouts. You can have a script that sets up your perfect AI development environment every time:
- Left pane: AI coding tool
- Right top pane: file watcher / test runner
- Right bottom pane: git status

We'll build this in Part 3.

### Real-World Example: The `cc` Alias Pattern

A popular pattern in the AI coding community (popularized by developers using Claude Code) is to create shell aliases that combine tmux with AI tools:

```bash
# Start (or reconnect to) an AI coding session for a project
cc() {
    local project="$1"
    if tmux has-session -t "cc-$project" 2>/dev/null; then
        tmux attach -t "cc-$project"
    else
        tmux new-session -s "cc-$project" -c "~/projects/$project"
    fi
}
```

Usage:
```bash
cc my-api          # Creates/attaches to tmux session "cc-my-api"
                   # in ~/projects/my-api directory
```

This gives you:
- One tmux session per project
- Named sessions so you can find them with `tmux ls`
- Automatic directory switching
- Reconnection if the session already exists

> **Source:** Pattern adapted from [Pedro Alonso's remote AI dev workflow](https://www.pedroalonso.net/blog/remote-ai-dev-workflow-claude-code-tmux-4090). Content was rephrased for compliance with licensing restrictions.

## 1.9 Installing tmux

### macOS

tmux is available via Homebrew:

```bash
brew install tmux
```

Verify the installation:

```bash
tmux -V
# Expected output: tmux 3.5a (or similar)
```

### Linux (Ubuntu/Debian)

```bash
sudo apt update && sudo apt install -y tmux
```

### Linux (Amazon Linux / RHEL / Fedora)

```bash
sudo yum install -y tmux    # Amazon Linux 2
sudo dnf install -y tmux    # Fedora / AL2023
```

### Verify It Works

```bash
# Start tmux
tmux

# You should see a green status bar at the bottom
# Type 'exit' or press Ctrl+d to close the session
exit
```

> **Note:** Only the latest tmux release is officially supported. As of early 2026, that's tmux 3.5a. If your package manager installs an older version, consider building from source — instructions are on the [tmux GitHub wiki](https://github.com/tmux/tmux/wiki/Installing).

## 1.10 Your First tmux Session — A Walkthrough

Let's put everything together with a step-by-step walkthrough. Open your terminal and follow along:

**Step 1: Create a named session**
```bash
tmux new -s learning
```
You're now inside tmux. Notice the green status bar showing `[learning] 0:zsh*`.

**Step 2: Run a command**
```bash
echo "Hello from tmux!"
```

**Step 3: Create a second window**
Press `C-b c` (Ctrl+b, then c). Notice the status bar now shows `0:zsh  1:zsh*` — you're in window 1.

**Step 4: Switch between windows**
Press `C-b 0` to go back to window 0. Press `C-b 1` to go to window 1. Press `C-b l` to toggle between the last two windows.

**Step 5: Split the window into panes**
Press `C-b %` to split vertically (left/right). You now have two panes. Press `C-b ←` and `C-b →` to move between them.

**Step 6: Detach**
Press `C-b d`. You're back in your regular terminal. tmux prints:
```
[detached (from session learning)]
```

**Step 7: Verify the session is still running**
```bash
tmux ls
# Output: learning: 2 windows (created Tue Apr 21 15:30:00 2026)
```

**Step 8: Reattach**
```bash
tmux attach -t learning
```
Everything is exactly as you left it — both windows, the split panes, your command history.

**Step 9: Clean up**
Press `C-b :` to open the command prompt, type `kill-session`, and press Enter. Or just type `exit` in each pane.

### 🤔 Think About It

You just used the core tmux workflow:
1. **Create** a session → `tmux new -s name`
2. **Work** inside it → windows, panes, commands
3. **Detach** → `C-b d`
4. **Reattach** → `tmux attach -t name`

This four-step cycle is 80% of what you'll do with tmux day-to-day. Everything else (configuration, plugins, scripting) builds on top of this foundation.

## 1.11 Visual Guide: What tmux Looks Like on Screen

Let's visualize what you actually see when using tmux with different layouts.

![Figure 6](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20340%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2222%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2215%22%20font-weight%3D%22700%22%3ESingle%20Window%2C%20No%20Splits%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Terminal%20frame%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%2240%22%20width%3D%22700%22%20height%3D%22260%22%20rx%3D%228%22%20fill%3D%22%231E1E1E%22%20stroke%3D%22%23444%22%20stroke-width%3D%222%22%2F%3E%0A%0A%20%20%3C%21--%20Title%20bar%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%2240%22%20width%3D%22700%22%20height%3D%2224%22%20rx%3D%228%22%20fill%3D%22%23333%22%2F%3E%0A%20%20%3Ccircle%20cx%3D%2270%22%20cy%3D%2252%22%20r%3D%226%22%20fill%3D%22%23FF5F56%22%2F%3E%0A%20%20%3Ccircle%20cx%3D%2288%22%20cy%3D%2252%22%20r%3D%226%22%20fill%3D%22%23FFBD2E%22%2F%3E%0A%20%20%3Ccircle%20cx%3D%22106%22%20cy%3D%2252%22%20r%3D%226%22%20fill%3D%22%2327C93F%22%2F%3E%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2256%22%20text-anchor%3D%22middle%22%20fill%3D%22%23999%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3ETerminal%20%E2%80%94%20tmux%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Content%20area%20--%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%2285%22%20fill%3D%22%234EC9B0%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E~%2Fprojects%2Fmy-api%20%24%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22260%22%20y%3D%2285%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3Els%20-la%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22103%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3Etotal%2048%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22119%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3Edrwxr-xr-x%20%208%20user%20staff%20%20256%20Apr%2021%2014%3A00%20.%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22135%22%20fill%3D%22%23569CD6%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E-rw-r--r--%20%201%20user%20staff%201234%20Apr%2021%2013%3A55%20main.go%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22151%22%20fill%3D%22%23569CD6%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E-rw-r--r--%20%201%20user%20staff%20%20567%20Apr%2021%2013%3A50%20go.mod%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22167%22%20fill%3D%22%234EC9B0%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3Edrwxr-xr-x%20%203%20user%20staff%20%20%2096%20Apr%2021%2013%3A45%20handlers%2F%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22195%22%20fill%3D%22%234EC9B0%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E~%2Fprojects%2Fmy-api%20%24%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22260%22%20y%3D%22195%22%20fill%3D%22%23666%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E%E2%96%8A%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Status%20bar%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%22276%22%20width%3D%22700%22%20height%3D%2224%22%20fill%3D%22%232EA043%22%2F%3E%0A%20%20%3Ctext%20x%3D%2260%22%20y%3D%22292%22%20fill%3D%22white%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3E%5Bwork%5D%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22110%22%20y%3D%22292%22%20fill%3D%22white%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E0%3Azsh%2A%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22660%22%20y%3D%22292%22%20fill%3D%22%23C8E6C9%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E14%3A30%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 7: tmux with a single window — looks like a normal terminal, but with a status bar*

![Figure 7](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20340%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2222%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2215%22%20font-weight%3D%22700%22%3ETwo%20Vertical%20Panes%20%28C-b%20%25%29%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Terminal%20frame%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%2240%22%20width%3D%22700%22%20height%3D%22260%22%20rx%3D%228%22%20fill%3D%22%231E1E1E%22%20stroke%3D%22%23444%22%20stroke-width%3D%222%22%2F%3E%0A%0A%20%20%3C%21--%20Title%20bar%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%2240%22%20width%3D%22700%22%20height%3D%2224%22%20rx%3D%228%22%20fill%3D%22%23333%22%2F%3E%0A%20%20%3Ccircle%20cx%3D%2270%22%20cy%3D%2252%22%20r%3D%226%22%20fill%3D%22%23FF5F56%22%2F%3E%0A%20%20%3Ccircle%20cx%3D%2288%22%20cy%3D%2252%22%20r%3D%226%22%20fill%3D%22%23FFBD2E%22%2F%3E%0A%20%20%3Ccircle%20cx%3D%22106%22%20cy%3D%2252%22%20r%3D%226%22%20fill%3D%22%2327C93F%22%2F%3E%0A%0A%20%20%3C%21--%20Left%20pane%20--%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%2285%22%20fill%3D%22%234EC9B0%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E%24%20vim%20main.go%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22103%22%20fill%3D%22%23569CD6%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3Epackage%20main%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22119%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3Eimport%20%22fmt%22%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22135%22%20fill%3D%22%23569CD6%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3Efunc%20main%28%29%20%26%23123%3B%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22151%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E%20%20fmt.Println%28%22hi%22%29%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22167%22%20fill%3D%22%23569CD6%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E%26%23125%3B%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Pane%20divider%20%28green%20%3D%20active%20pane%20is%20left%29%20--%3E%0A%20%20%3Cline%20x1%3D%22399%22%20y1%3D%2264%22%20x2%3D%22399%22%20y2%3D%22276%22%20stroke%3D%22%232EA043%22%20stroke-width%3D%222%22%2F%3E%0A%0A%20%20%3C%21--%20Right%20pane%20--%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%2285%22%20fill%3D%22%234EC9B0%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E%24%20go%20test%20.%2F...%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22103%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3Eok%20%20%20my-api%2Fhandlers%200.3s%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22119%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3Eok%20%20%20my-api%2Fmodels%20%20%200.1s%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22135%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22151%22%20fill%3D%22%234EC9B0%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E%24%20%E2%96%8A%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Status%20bar%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%22276%22%20width%3D%22700%22%20height%3D%2224%22%20fill%3D%22%232EA043%22%2F%3E%0A%20%20%3Ctext%20x%3D%2260%22%20y%3D%22292%22%20fill%3D%22white%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3E%5Bwork%5D%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22110%22%20y%3D%22292%22%20fill%3D%22white%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E0%3Acode%2A%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22660%22%20y%3D%22292%22%20fill%3D%22%23C8E6C9%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E14%3A35%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 8: Two vertical panes — edit code on the left, run tests on the right*

![Figure 8](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20380%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2222%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2215%22%20font-weight%3D%22700%22%3EAI%20Dev%20Layout%3A%203%20Panes%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Terminal%20frame%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%2240%22%20width%3D%22700%22%20height%3D%22300%22%20rx%3D%228%22%20fill%3D%22%231E1E1E%22%20stroke%3D%22%23444%22%20stroke-width%3D%222%22%2F%3E%0A%0A%20%20%3C%21--%20Title%20bar%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%2240%22%20width%3D%22700%22%20height%3D%2224%22%20rx%3D%228%22%20fill%3D%22%23333%22%2F%3E%0A%20%20%3Ccircle%20cx%3D%2270%22%20cy%3D%2252%22%20r%3D%226%22%20fill%3D%22%23FF5F56%22%2F%3E%0A%20%20%3Ccircle%20cx%3D%2288%22%20cy%3D%2252%22%20r%3D%226%22%20fill%3D%22%23FFBD2E%22%2F%3E%0A%20%20%3Ccircle%20cx%3D%22106%22%20cy%3D%2252%22%20r%3D%226%22%20fill%3D%22%2327C93F%22%2F%3E%0A%0A%20%20%3C%21--%20Left%20pane%20%28large%29%20-%20AI%20tool%20--%3E%0A%20%20%3Crect%20x%3D%2255%22%20y%3D%2268%22%20width%3D%22340%22%20height%3D%22248%22%20fill%3D%22%231a1a2e%22%2F%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%2288%22%20fill%3D%22%238C4FFF%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3E%E2%95%AD%E2%94%80%20Claude%20Code%20%E2%94%80%E2%94%80%E2%94%80%E2%94%80%E2%94%80%E2%94%80%E2%94%80%E2%94%80%E2%94%80%E2%94%80%E2%94%80%E2%94%80%E2%94%80%E2%95%AE%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22106%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EAnalyzing%20handlers%2Fuser.go...%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22122%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E%E2%9C%93%20Added%20error%20handling%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22138%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E%E2%9C%93%20Updated%20tests%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22154%22%20fill%3D%22%23FF9900%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E%E2%9F%B3%20Running%20go%20test%20.%2F...%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Pane%20divider%20vertical%20--%3E%0A%20%20%3Cline%20x1%3D%22399%22%20y1%3D%2268%22%20x2%3D%22399%22%20y2%3D%22316%22%20stroke%3D%22%232EA043%22%20stroke-width%3D%222%22%2F%3E%0A%0A%20%20%3C%21--%20Right%20top%20pane%20-%20test%20output%20--%3E%0A%20%20%3Crect%20x%3D%22403%22%20y%3D%2268%22%20width%3D%22343%22%20height%3D%22124%22%20fill%3D%22%231E1E1E%22%2F%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%2288%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E%E2%94%80%E2%94%80%20test%20output%20%E2%94%80%E2%94%80%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22106%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EPASS%3A%20TestCreateUser%20%280.02s%29%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22122%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EPASS%3A%20TestGetUser%20%280.01s%29%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22138%22%20fill%3D%22%23D13212%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EFAIL%3A%20TestDeleteUser%20%280.03s%29%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22154%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E%20%20expected%20204%2C%20got%20403%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Pane%20divider%20horizontal%20--%3E%0A%20%20%3Cline%20x1%3D%22403%22%20y1%3D%22194%22%20x2%3D%22746%22%20y2%3D%22194%22%20stroke%3D%22%23444%22%20stroke-width%3D%221%22%2F%3E%0A%0A%20%20%3C%21--%20Right%20bottom%20pane%20-%20git%20--%3E%0A%20%20%3Crect%20x%3D%22403%22%20y%3D%22196%22%20width%3D%22343%22%20height%3D%22120%22%20fill%3D%22%231E1E1E%22%2F%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22216%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E%E2%94%80%E2%94%80%20git%20status%20%E2%94%80%E2%94%80%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22234%22%20fill%3D%22%23D13212%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EM%20%20handlers%2Fuser.go%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22250%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EA%20%20handlers%2Fuser_test.go%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22415%22%20y%3D%22266%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E2%20files%20changed%2C%2045%20insertions%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Status%20bar%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%22316%22%20width%3D%22700%22%20height%3D%2224%22%20fill%3D%22%232EA043%22%2F%3E%0A%20%20%3Ctext%20x%3D%2260%22%20y%3D%22332%22%20fill%3D%22white%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3E%5Bcc-my-api%5D%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22155%22%20y%3D%22332%22%20fill%3D%22white%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E0%3Adev%2A%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22660%22%20y%3D%22332%22%20fill%3D%22%23C8E6C9%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E14%3A42%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 9: A typical AI development layout — AI tool on the left, tests and git on the right*

### Layout Patterns for Different Workflows

| Layout | Use Case | How to Create |
|--------|----------|---------------|
| Single pane | Simple tasks, reading docs | Default (no splits) |
| 2 vertical panes | Code + terminal side by side | `C-b %` |
| 2 horizontal panes | Main task + logs below | `C-b "` |
| 3 panes (main + 2 small) | AI tool + tests + git | `C-b %` then `C-b "` in right pane |
| 4 panes (grid) | Monitoring multiple things | `C-b %` then `C-b "` in each half |

> **Pro tip:** `C-b z` zooms any pane to full screen temporarily. Press `C-b z` again to unzoom. This is incredibly useful when you need to focus on one pane's output.

## 1.12 Part 1 Summary — Key Takeaways

You've covered a lot of ground. Here's what you should now understand:

### Concepts

| Concept | Key Point |
|---------|-----------|
| Terminal multiplexer | A program that manages multiple terminals inside one window |
| tmux | The industry-standard terminal multiplexer, created in 2007 by Nicholas Marriott |
| Client-server model | tmux server runs in background; your terminal is a client that attaches to it |
| Session | A workspace containing windows — survives disconnections |
| Window | A tab within a session |
| Pane | A split within a window running one program |
| Prefix key | `Ctrl+b` by default — tells tmux the next key is a command |
| Detach/Attach | The killer feature — disconnect and reconnect without losing state |
| Status bar | The green bar showing session name, windows, and time |

### Commands You Should Know

| Command / Key | What It Does |
|---------------|-------------|
| `tmux new -s name` | Create a named session |
| `tmux attach -t name` | Attach to a named session |
| `tmux new -As name` | Attach or create (the smart one-liner) |
| `tmux ls` | List all sessions |
| `C-b d` | Detach |
| `C-b c` | New window |
| `C-b n` / `C-b p` | Next / previous window |
| `C-b 0-9` | Go to window by number |
| `C-b %` | Split vertically |
| `C-b "` | Split horizontally |
| `C-b ←↑↓→` | Move between panes |
| `C-b z` | Zoom/unzoom pane |
| `C-b ?` | Show all key bindings |

### Why It Matters for AI Tools

- **Session persistence** — fire-and-forget AI tasks
- **Parallel sessions** — multiple AI agents at once
- **Scrollback** — review long AI output
- **Multi-device** — start on laptop, check from phone
- **Scriptable layouts** — reproducible dev environments

---

### What's Next

**Part 2: Core Concepts & Workflows** will cover:
- Building a `.tmux.conf` from scratch (making tmux actually pleasant to use)
- Copy mode and scrollback (searching through output)
- Plugins with TPM (tmux plugin manager)
- Advanced session/window/pane management
- Scripting tmux for automation
- Integration patterns with Kiro CLI and Claude Code

---

> 📚 **Sources used in this part:**
> - [tmux GitHub Wiki — Getting Started](https://github.com/tmux/tmux/wiki/Getting-Started) (official documentation by Nicholas Marriott)
> - [Pedro Alonso — Remote AI Dev Workflow](https://www.pedroalonso.net/blog/remote-ai-dev-workflow-claude-code-tmux-4090)
> - [Wikipedia — tmux](https://en.wikipedia.org/wiki/Tmux)
> - Various community guides and comparisons (content was rephrased for compliance with licensing restrictions)

---

# Part 2: Core Concepts & Workflows

> **Target study time: ~2 hours** · This part transforms tmux from "I know what it is" to "I use it productively every day."

---

## 2.1 Building Your `.tmux.conf` From Scratch

tmux's defaults are functional but awkward. The configuration file `~/.tmux.conf` is where you make tmux yours. It's loaded once when the tmux server starts.

> **Important:** `.tmux.conf` runs when the *server* starts, not when you create a new session. To reload after editing:
> - From inside tmux: `C-b :` then type `source ~/.tmux.conf`
> - Or add a reload keybinding (we will below)

Let's build a config line by line. Each setting includes *why* it matters, not just *what* it does.

### The Prefix Key — Remapping to `Ctrl+a`

The default `Ctrl+b` requires an awkward hand stretch. `Ctrl+a` (GNU Screen's default) is much more natural — your pinky stays on `Ctrl` and your ring finger taps `a`.

```bash
# ~/.tmux.conf

# Change prefix from Ctrl+b to Ctrl+a
set -g prefix C-a
unbind C-b
bind C-a send-prefix
```

**What each line does:**
- `set -g prefix C-a` — sets the new prefix globally
- `unbind C-b` — removes the old prefix binding
- `bind C-a send-prefix` — pressing `C-a C-a` sends a literal `Ctrl+a` to the shell (needed for beginning-of-line in bash)

> ⚠️ **For this notebook:** All keybindings from here on use `C-a` as the prefix. If you keep the default, substitute `C-b` everywhere.

### Why Not `Ctrl+Space`?

Some people use `C-Space` as prefix. It works well but can conflict with input method editors (IME) on macOS for non-English keyboards. `C-a` is the safest choice for most users.

### Sensible Defaults

These settings fix tmux's most annoying default behaviors:

```bash
# Start window and pane numbering from 1 (not 0)
set -g base-index 1
setw -g pane-base-index 1

# Renumber windows when one is closed (no gaps: 1,2,3 not 1,3,4)
set -g renumber-windows on

# Increase scrollback buffer (default is 2000 lines)
set -g history-limit 50000

# Reduce escape delay (important for vim/neovim users)
set -sg escape-time 0

# Enable mouse support (scrolling, clicking panes, resizing)
set -g mouse on

# Don't auto-rename windows (keep your custom names)
set -g allow-rename off

# Enable 256 colors and true color
set -g default-terminal "tmux-256color"
set -ag terminal-overrides ",xterm-256color:RGB"

# Display messages for 3 seconds (default is too short)
set -g display-time 3000

# Display pane numbers for 3 seconds (more time to pick one)
set -g display-panes-time 3000
```

**Why `base-index 1`?** Your keyboard's number row starts at 1, not 0. With `base-index 1`, window 1 is `C-a 1`, window 2 is `C-a 2`, etc. — matching the physical key positions.

**Why `escape-time 0`?** tmux waits after receiving `Escape` to see if it's part of a longer sequence. This adds noticeable lag in vim. Setting it to 0 eliminates the delay.

**Why `history-limit 50000`?** AI tools produce massive output. 2000 lines (the default) fills up fast. 50K gives you room to scroll back through long sessions.

### Better Key Bindings for Splits and Navigation

The default split keys (`%` and `"`) are impossible to remember. Let's replace them with intuitive alternatives:

```bash
# Intuitive split keys (| for vertical, - for horizontal)
bind | split-window -h -c "#{pane_current_path}"
bind - split-window -v -c "#{pane_current_path}"
unbind %
unbind '"'
```

**The `-c "#{pane_current_path}"` part** is crucial — it makes new panes open in the same directory as the current pane. Without it, new panes always open in your home directory.

```bash
# Also create new windows in the current path
bind c new-window -c "#{pane_current_path}"

# Easy config reload
bind r source-file ~/.tmux.conf \; display "Config reloaded!"

# Switch panes with Alt+arrow (no prefix needed!)
bind -n M-Left select-pane -L
bind -n M-Right select-pane -R
bind -n M-Up select-pane -U
bind -n M-Down select-pane -D

# Switch windows with Shift+arrow (no prefix needed!)
bind -n S-Left previous-window
bind -n S-Right next-window

# Resize panes with prefix + Shift+arrow
bind -r S-Left resize-pane -L 5
bind -r S-Right resize-pane -R 5
bind -r S-Up resize-pane -U 5
bind -r S-Down resize-pane -D 5
```

**The `-n` flag** means "no prefix needed" — the key works directly from the `root` table. So `Alt+Left` immediately switches panes without pressing `C-a` first.

**The `-r` flag** means "repeatable" — after pressing the prefix once, you can press the key multiple times to keep resizing without re-pressing the prefix.

![Figure 9](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20260%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2222%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2215%22%20font-weight%3D%22700%22%3ECustom%20Keybindings%3A%20Before%20vs%20After%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Before%20--%3E%0A%20%20%3Crect%20x%3D%2220%22%20y%3D%2240%22%20width%3D%22370%22%20height%3D%22200%22%20rx%3D%2212%22%20fill%3D%22%23FFFFFF%22%20stroke%3D%22%23D5DBDB%22%20stroke-width%3D%221%22%20stroke-dasharray%3D%226%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%2232%22%20y%3D%2260%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3E%E2%9D%8C%20Default%20Keys%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2240%22%20y%3D%2280%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3EC-b%20%25%20%20%20%20%E2%86%92%20vertical%20split%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2240%22%20y%3D%22100%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3EC-b%20%22%20%20%20%20%E2%86%92%20horizontal%20split%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2240%22%20y%3D%22120%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3EC-b%20%E2%86%90%20%20%20%20%E2%86%92%20move%20to%20left%20pane%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2240%22%20y%3D%22140%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3EC-b%20n%20%20%20%20%E2%86%92%20next%20window%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2240%22%20y%3D%22160%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3EC-b%20p%20%20%20%20%E2%86%92%20previous%20window%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2240%22%20y%3D%22185%22%20fill%3D%22%23D13212%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3EUnintuitive%2C%20always%20needs%20prefix%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20After%20--%3E%0A%20%20%3Crect%20x%3D%22410%22%20y%3D%2240%22%20width%3D%22370%22%20height%3D%22200%22%20rx%3D%2212%22%20fill%3D%22%23FFFFFF%22%20stroke%3D%22%23D5DBDB%22%20stroke-width%3D%221%22%20stroke-dasharray%3D%226%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%22422%22%20y%3D%2260%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3E%E2%9C%85%20Custom%20Keys%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22430%22%20y%3D%2280%22%20fill%3D%22%23232F3E%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3EC-a%20%7C%20%20%20%20%E2%86%92%20vertical%20split%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22430%22%20y%3D%22100%22%20fill%3D%22%23232F3E%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3EC-a%20-%20%20%20%20%E2%86%92%20horizontal%20split%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22430%22%20y%3D%22120%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EAlt%2B%E2%86%90%20%20%20%20%E2%86%92%20move%20to%20left%20pane%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22430%22%20y%3D%22140%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EShift%2B%E2%86%92%20%20%E2%86%92%20next%20window%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22430%22%20y%3D%22160%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EShift%2B%E2%86%90%20%20%E2%86%92%20previous%20window%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22430%22%20y%3D%22185%22%20fill%3D%22%232EA043%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3EIntuitive%2C%20pane%20nav%20needs%20no%20prefix%21%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 10: Custom keybindings make tmux feel natural*

### Vi Mode for Copy and Command Prompt

If you use vim (or want vim-style navigation in copy mode):

```bash
# Use vi keys in copy mode and command prompt
set -g mode-keys vi
set -g status-keys vi

# Vi-style copy mode bindings
bind -T copy-mode-vi v send -X begin-selection
bind -T copy-mode-vi y send -X copy-selection-and-cancel

# Copy to system clipboard (macOS)
bind -T copy-mode-vi y send -X copy-pipe-and-cancel "pbcopy"

# Copy to system clipboard (Linux with xclip)
# bind -T copy-mode-vi y send -X copy-pipe-and-cancel "xclip -selection clipboard"
```

With these settings, copy mode works like vim:
- Enter copy mode: `C-a [`
- Navigate: `h j k l` (or arrow keys)
- Start selection: `v`
- Copy: `y` (also copies to system clipboard)
- Search forward: `/`
- Search backward: `?`
- Paste: `C-a ]`

### Status Bar Customization

The default green bar is functional but plain. Here's a clean, informative status bar:

```bash
# Status bar position
set -g status-position bottom

# Status bar colors
set -g status-style 'bg=#1E1E1E,fg=#D4D4D4'

# Left side: session name
set -g status-left-length 30
set -g status-left '#[fg=#FF9900,bold] #S #[default]│ '

# Right side: date and time
set -g status-right '#[fg=#687078]%H:%M  %d-%b-%Y '

# Window list styling
set -g window-status-format '#[fg=#687078] #I:#W '
set -g window-status-current-format '#[fg=#527FFF,bold] #I:#W* '
set -g window-status-separator ''

# Pane borders
set -g pane-border-style 'fg=#444444'
set -g pane-active-border-style 'fg=#527FFF'
```

**Format variables explained:**
- `#S` — session name
- `#I` — window index
- `#W` — window name
- `%H:%M` — time (hours:minutes)
- `%d-%b-%Y` — date (21-Apr-2026)

![Figure 10](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20100%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2220%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22700%22%3ECustom%20Status%20Bar%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Status%20bar%20background%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%2235%22%20width%3D%22700%22%20height%3D%2232%22%20rx%3D%224%22%20fill%3D%22%231E1E1E%22%20stroke%3D%22%23333%22%20stroke-width%3D%221%22%2F%3E%0A%0A%20%20%3C%21--%20Session%20name%20%28orange%29%20--%3E%0A%20%20%3Ctext%20x%3D%2265%22%20y%3D%2256%22%20fill%3D%22%23FF9900%22%20font-family%3D%22monospace%22%20font-size%3D%2213%22%20font-weight%3D%22700%22%3E%20work%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22120%22%20y%3D%2256%22%20fill%3D%22%23555%22%20font-family%3D%22monospace%22%20font-size%3D%2213%22%3E%E2%94%82%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Windows%20--%3E%0A%20%20%3Ctext%20x%3D%22135%22%20y%3D%2256%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E%201%3Ashell%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22215%22%20y%3D%2256%22%20fill%3D%22%23527FFF%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%20font-weight%3D%22700%22%3E%202%3Avim%2A%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22285%22%20y%3D%2256%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E%203%3Alogs%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Right%20side%20--%3E%0A%20%20%3Ctext%20x%3D%22630%22%20y%3D%2256%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E15%3A30%20%2021-Apr-2026%3C%2Ftext%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2285%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EDark%20theme%20with%20orange%20session%20name%2C%20blue%20active%20window%2C%20muted%20inactive%20windows%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 11: A clean, dark-themed custom status bar*

### The Complete Starter `.tmux.conf`

Here's everything combined into a single, copy-paste-ready config:

```bash
# ─── Prefix ───
set -g prefix C-a
unbind C-b
bind C-a send-prefix

# ─── Sensible Defaults ───
set -g base-index 1
setw -g pane-base-index 1
set -g renumber-windows on
set -g history-limit 50000
set -sg escape-time 0
set -g mouse on
set -g allow-rename off
set -g default-terminal "tmux-256color"
set -ag terminal-overrides ",xterm-256color:RGB"
set -g display-time 3000
set -g display-panes-time 3000
set -g focus-events on

# ─── Key Bindings ───
bind | split-window -h -c "#{pane_current_path}"
bind - split-window -v -c "#{pane_current_path}"
unbind %
unbind '"'
bind c new-window -c "#{pane_current_path}"
bind r source-file ~/.tmux.conf \; display "Config reloaded!"

# Pane navigation (no prefix)
bind -n M-Left select-pane -L
bind -n M-Right select-pane -R
bind -n M-Up select-pane -U
bind -n M-Down select-pane -D

# Window navigation (no prefix)
bind -n S-Left previous-window
bind -n S-Right next-window

# Pane resizing (repeatable)
bind -r S-Left resize-pane -L 5
bind -r S-Right resize-pane -R 5
bind -r S-Up resize-pane -U 5
bind -r S-Down resize-pane -D 5

# ─── Vi Mode ───
set -g mode-keys vi
set -g status-keys vi
bind -T copy-mode-vi v send -X begin-selection
bind -T copy-mode-vi y send -X copy-pipe-and-cancel "pbcopy"

# ─── Status Bar ───
set -g status-position bottom
set -g status-style 'bg=#1E1E1E,fg=#D4D4D4'
set -g status-left-length 30
set -g status-left '#[fg=#FF9900,bold] #S #[default]│ '
set -g status-right '#[fg=#687078]%H:%M  %d-%b-%Y '
set -g window-status-format '#[fg=#687078] #I:#W '
set -g window-status-current-format '#[fg=#527FFF,bold] #I:#W* '
set -g window-status-separator ''
set -g pane-border-style 'fg=#444444'
set -g pane-active-border-style 'fg=#527FFF'
```

> **To install:** Copy this into `~/.tmux.conf`, then either restart tmux or press `C-a r` (after the first load with `C-b :source ~/.tmux.conf`).

### 🤔 Common Misconception

**"I need to restart tmux to apply config changes"**

No! You can reload the config at any time with `source-file`:
- From the command prompt: `C-a :` then `source ~/.tmux.conf`
- With our keybinding: `C-a r`

The config is applied immediately. However, some settings (like `default-terminal`) only take effect for *new* sessions/windows.

## 2.2 Copy Mode & Scrollback — Searching Through Output

Copy mode is one of tmux's most powerful features, and one of the least understood by beginners. It lets you scroll through terminal output, search for text, and copy content — all without a mouse.

### Entering and Exiting Copy Mode

| Action | Key (with our config) |
|--------|----------------------|
| Enter copy mode | `C-a [` |
| Exit copy mode | `q` |
| Also exits | `Escape` |

When you enter copy mode, the pane "freezes" — new output is buffered but not shown until you exit. A line counter appears in the top-right corner showing your position.

![Figure 11](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20280%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%20%20%3Cdefs%3E%3Cmarker%20id%3D%22arrowhead%22%20markerWidth%3D%2210%22%20markerHeight%3D%227%22%20refX%3D%2210%22%20refY%3D%223.5%22%20orient%3D%22auto%22%3E%3Cpolygon%20points%3D%220%200%2C%2010%203.5%2C%200%207%22%20fill%3D%22%23687078%22%2F%3E%3C%2Fmarker%3E%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2222%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2215%22%20font-weight%3D%22700%22%3ECopy%20Mode%20Workflow%20%28vi%20keys%29%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Steps%20--%3E%0A%20%20%3Crect%20x%3D%2230%22%20y%3D%2250%22%20width%3D%22130%22%20height%3D%2250%22%20rx%3D%2210%22%20fill%3D%22%23527FFF%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%2295%22%20y%3D%2280%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22600%22%3EC-a%20%5B%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2295%22%20y%3D%22120%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3EEnter%20copy%20mode%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22160%22%20y1%3D%2275%22%20x2%3D%22200%22%20y2%3D%2275%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%0A%20%20%3Crect%20x%3D%22200%22%20y%3D%2250%22%20width%3D%22130%22%20height%3D%2250%22%20rx%3D%2210%22%20fill%3D%22%2300A1C9%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22265%22%20y%3D%2280%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22600%22%3ENavigate%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22265%22%20y%3D%22120%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3Eh%20j%20k%20l%20%2F%20arrows%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22265%22%20y%3D%22133%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3E%2F%20to%20search%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22330%22%20y1%3D%2275%22%20x2%3D%22370%22%20y2%3D%2275%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%0A%20%20%3Crect%20x%3D%22370%22%20y%3D%2250%22%20width%3D%22130%22%20height%3D%2250%22%20rx%3D%2210%22%20fill%3D%22%23FF9900%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22435%22%20y%3D%2280%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22600%22%3Ev%20%28select%29%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22435%22%20y%3D%22120%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3EStart%20selection%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22435%22%20y%3D%22133%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3EMove%20to%20expand%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22500%22%20y1%3D%2275%22%20x2%3D%22540%22%20y2%3D%2275%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%0A%20%20%3Crect%20x%3D%22540%22%20y%3D%2250%22%20width%3D%22130%22%20height%3D%2250%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22605%22%20y%3D%2280%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22600%22%3Ey%20%28copy%29%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22605%22%20y%3D%22120%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3ECopy%20%2B%20exit%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22605%22%20y%3D%22133%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3E%E2%86%92%20clipboard%3C%2Ftext%3E%0A%0A%20%20%3Cline%20x1%3D%22670%22%20y1%3D%2275%22%20x2%3D%22710%22%20y2%3D%2275%22%20stroke%3D%22%23687078%22%20stroke-width%3D%222%22%20marker-end%3D%22url%28%23arrowhead%29%22%2F%3E%0A%0A%20%20%3Crect%20x%3D%22710%22%20y%3D%2250%22%20width%3D%2270%22%20height%3D%2250%22%20rx%3D%2210%22%20fill%3D%22%238C4FFF%22%20filter%3D%22url%28%23shadow%29%22%20opacity%3D%220.9%22%2F%3E%0A%20%20%3Ctext%20x%3D%22745%22%20y%3D%2280%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3EC-a%20%5D%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22745%22%20y%3D%22120%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3EPaste%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Navigation%20reference%20--%3E%0A%20%20%3Crect%20x%3D%2230%22%20y%3D%22155%22%20width%3D%22740%22%20height%3D%22110%22%20rx%3D%2212%22%20fill%3D%22%23F2F3F3%22%20stroke%3D%22%23D5DBDB%22%20stroke-width%3D%221%22%20stroke-dasharray%3D%226%2C3%22%2F%3E%0A%20%20%3Ctext%20x%3D%2242%22%20y%3D%22175%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EVi-mode%20Navigation%20in%20Copy%20Mode%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2250%22%20y%3D%22190%22%20fill%3D%22%23232F3E%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3Eh%2Fj%2Fk%2Fl%20%20%20%20Move%20cursor%20%20%20%20%20%20%20%20%20%20g%20%20%20%20Jump%20to%20top%20%20%20%20%20%20%20%20%20%2Fpattern%20%20%20Search%20forward%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2250%22%20y%3D%22210%22%20fill%3D%22%23232F3E%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3EC-u%2FC-d%20%20%20%20Half-page%20up%2Fdown%20%20%20%20%20G%20%20%20%20Jump%20to%20bottom%20%20%20%20%20%20%3Fpattern%20%20%20Search%20backward%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2250%22%20y%3D%22230%22%20fill%3D%22%23232F3E%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3Ew%2Fb%20%20%20%20%20%20%20%20Next%2Fprev%20word%20%20%20%20%20%20%20%200%2F%24%20%20Start%2Fend%20of%20line%20%20%20n%2FN%20%20%20%20%20%20%20%20Next%2Fprev%20match%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2250%22%20y%3D%22250%22%20fill%3D%22%23232F3E%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3Ev%20%20%20%20%20%20%20%20%20%20Begin%20selection%20%20%20%20%20%20%20%20y%20%20%20%20Copy%20selection%20%20%20%20%20%20%20q%20%20%20%20%20%20%20%20%20%20Exit%20copy%20mode%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 12: Copy mode workflow — navigate, select, copy, paste*

### Searching in Copy Mode

This is where copy mode becomes essential for AI tool workflows. When Claude Code or Kiro CLI produces pages of output, you need to find specific parts.

**Search forward:** Press `/`, type your search term, press `Enter`
**Search backward:** Press `?`, type your search term, press `Enter`
**Next match:** `n`
**Previous match:** `N`

The search is case-sensitive by default. The matched text is highlighted.

### Practical Example: Finding an Error in AI Output

1. AI tool produces 500 lines of output including a test failure
2. Press `C-a [` to enter copy mode
3. Press `/` and type `FAIL` then `Enter`
4. Press `n` to jump between matches
5. Found it! Press `v` to start selecting the error message
6. Move down with `j` to select multiple lines
7. Press `y` to copy (goes to system clipboard with our config)
8. Paste into a chat, ticket, or another pane with `C-a ]`

### The Scrollback Buffer

Every pane has a scrollback buffer — a history of all output. With our config (`history-limit 50000`), each pane stores the last 50,000 lines.

In copy mode, you can scroll through this entire history:
- `Page Up` / `C-u` — scroll up half a page
- `Page Down` / `C-d` — scroll down half a page
- `g` — jump to the very top (oldest output)
- `G` — jump to the very bottom (newest output)

> **Pro tip:** With mouse enabled (`set -g mouse on`), you can also scroll with your trackpad/mouse wheel. This automatically enters copy mode.

### 🤔 Think About It

You're running Kiro CLI in a tmux pane. It generates a long response with a code block you want to copy. Without copy mode, you'd have to:
1. Scroll up with the mouse (if your terminal supports it)
2. Click and drag to select (often selects across pane borders)
3. Hope the selection is right

With copy mode:
1. `C-a [` → enter copy mode
2. `/def handle` → search for the function
3. `v` → start selection
4. `}` → jump to end of block (vi motion)
5. `y` → copy to clipboard

Precise, keyboard-driven, works over SSH. This is why tmux copy mode matters.

## 2.3 Plugins with TPM (Tmux Plugin Manager)

tmux has a thriving plugin ecosystem managed by **TPM** (Tmux Plugin Manager). Plugins extend tmux with features like session persistence across reboots, better copy/paste, and status bar enhancements.

### Installing TPM

```bash
# Clone TPM into the tmux plugins directory
git clone https://github.com/tmux-plugins/tpm ~/.tmux/plugins/tpm
```

Add this to the **bottom** of your `~/.tmux.conf`:

```bash
# ─── Plugins ───
set -g @plugin 'tmux-plugins/tpm'
set -g @plugin 'tmux-plugins/tmux-sensible'
set -g @plugin 'tmux-plugins/tmux-resurrect'
set -g @plugin 'tmux-plugins/tmux-continuum'
set -g @plugin 'tmux-plugins/tmux-yank'

# Initialize TPM (keep this line at the very bottom)
run '~/.tmux/plugins/tpm/tpm'
```

Then reload tmux (`C-a r`) and press `C-a I` (capital I) to install plugins. TPM will clone them and show a success message.

### Key TPM Commands

| Keys | Action |
|------|--------|
| `C-a I` | Install new plugins |
| `C-a U` | Update plugins |
| `C-a alt+u` | Uninstall removed plugins |

### The Essential Plugins

**tmux-resurrect** — Save and restore tmux sessions across system restarts

This is the plugin that makes tmux truly persistent. Without it, rebooting your machine kills all tmux sessions. With it:
- `C-a Ctrl+s` — save all sessions, windows, panes, and running programs
- `C-a Ctrl+r` — restore everything exactly as it was

It saves: session names, window layouts, pane directories, and even running programs (vim, ssh, etc.).

**tmux-continuum** — Automatic saving (pairs with resurrect)

```bash
# Auto-save every 15 minutes
set -g @continuum-save-interval '15'

# Auto-restore when tmux server starts
set -g @continuum-restore 'on'
```

With both plugins, your tmux environment survives reboots automatically. Start your machine, open a terminal, run `tmux` — and everything is back.

**tmux-yank** — Better clipboard integration

Enhances copy mode to work seamlessly with the system clipboard on macOS, Linux (X11 and Wayland), and WSL. With tmux-yank, `y` in copy mode always copies to the system clipboard.

**tmux-sensible** — Universal sensible defaults

A set of options that everyone agrees on (UTF-8, increased history, etc.). It's a safety net — if you forget a setting, tmux-sensible probably has it.

### 🤔 Common Misconception

**"Plugins make tmux slow"**

TPM plugins are shell scripts that run once at startup. They don't add runtime overhead. The only plugin that does periodic work is tmux-continuum (auto-save every 15 minutes), and that's a lightweight file write.

If tmux feels slow, it's almost always the `escape-time` setting (set it to 0) or a misconfigured status bar running expensive shell commands.

## 2.4 Advanced Session, Window & Pane Management

Now that you have a solid config, let's go deeper into managing your tmux environment.

### Session Management Patterns

**One session per project** is the most common pattern:

```bash
tmux new -As my-api        # API project
tmux new -As infra          # Infrastructure work
tmux new -As learning       # Study/learning
```

Switch between sessions from inside tmux:
- `C-a s` — session picker (tree mode)
- `C-a (` — previous session
- `C-a )` — next session

**Tree mode (`C-a s`)** is the most powerful way to navigate. It shows all sessions, windows, and panes in a tree. Use arrow keys to navigate, `Enter` to switch, `x` to kill.

![Figure 12](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20300%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2222%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2215%22%20font-weight%3D%22700%22%3ETree%20Mode%20%28C-a%20s%29%20%E2%80%94%20Session%20Navigator%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Tree%20mode%20panel%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%2240%22%20width%3D%22700%22%20height%3D%22240%22%20rx%3D%226%22%20fill%3D%22%231E1E1E%22%20stroke%3D%22%23444%22%20stroke-width%3D%221%22%2F%3E%0A%0A%20%20%3C%21--%20Tree%20entries%20--%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%2270%22%20fill%3D%22%23FF9900%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%20font-weight%3D%22700%22%3E%280%29%20%E2%96%BC%20my-api%3A%203%20windows%20%28attached%29%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22100%22%20y%3D%2290%22%20fill%3D%22%23527FFF%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E%20%20%281%29%201%3A%20code%2A%20%282%20panes%29%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22100%22%20y%3D%22108%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E%20%20%282%29%202%3A%20tests%20%281%20pane%29%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22100%22%20y%3D%22126%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E%20%20%283%29%203%3A%20logs%20%281%20pane%29%3C%2Ftext%3E%0A%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22152%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E%284%29%20%E2%96%B6%20infra%3A%202%20windows%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22174%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3E%285%29%20%E2%96%B6%20learning%3A%201%20windows%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Divider%20--%3E%0A%20%20%3Cline%20x1%3D%2250%22%20y1%3D%22195%22%20x2%3D%22750%22%20y2%3D%22195%22%20stroke%3D%22%23444%22%20stroke-width%3D%221%22%2F%3E%0A%0A%20%20%3C%21--%20Preview%20area%20--%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22215%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3EPreview%3A%20my-api%20%3E%201%3A%20code%20%3E%20pane%200%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22235%22%20fill%3D%22%234EC9B0%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E~%2Fprojects%2Fmy-api%20%24%20vim%20main.go%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22253%22%20fill%3D%22%23569CD6%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3Epackage%20main%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Key%20hints%20--%3E%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%22275%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2210%22%3E%E2%86%91%E2%86%93%20navigate%20%20%E2%86%92expand%20%20%E2%86%90collapse%20%20Enter%3Dswitch%20%20x%3Dkill%20%20q%3Dexit%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 13: Tree mode — your command center for navigating sessions, windows, and panes*

### Window Management Deep Dive

Beyond the basics, here are window operations you'll use regularly:

| Keys | Action | Notes |
|------|--------|-------|
| `C-a c` | New window | Opens in current directory (with our config) |
| `C-a ,` | Rename window | Give it a meaningful name |
| `C-a &` | Kill window | Prompts for confirmation |
| `C-a w` | Window picker | Like tree mode but starts at window level |
| `C-a l` | Last window | Toggle between current and previous |
| `C-a .` | Move window | Change window's index number |

**Naming convention tip:** Name windows by what you're doing, not what program is running:
- ✅ `code`, `tests`, `logs`, `deploy`
- ❌ `bash`, `zsh`, `vim` (these are the defaults and tell you nothing)

### Pane Management Deep Dive

| Keys | Action |
|------|--------|
| `C-a z` | Zoom pane (toggle full-screen) |
| `C-a !` | Break pane into its own window |
| `C-a q` | Show pane numbers (press number to switch) |
| `C-a {` | Swap pane with previous |
| `C-a }` | Swap pane with next |
| `C-a Space` | Cycle through layouts |
| `C-a x` | Kill pane |

**Zoom (`C-a z`)** is one of the most underused features. When you need to focus on one pane's output (e.g., reading a long AI response), zoom it to full screen. Press `C-a z` again to restore the layout. The status bar shows `Z` when a pane is zoomed.

**Break pane (`C-a !`)** is useful when a pane grows important enough to deserve its own window. For example, you split a pane to run a quick command, but then that command becomes a long-running process — break it out into its own window.

### The `join-pane` Command — Moving Panes Between Windows

Sometimes you want to move a pane from one window to another. The `join-pane` command does this:

```bash
# From the command prompt (C-a :)
# Move pane from window 3 to the current window
:join-pane -s :3

# Move the current pane to window 2
:join-pane -t :2
```

You can also use the **mark and swap** pattern:
1. Go to the pane you want to move
2. `C-a m` — mark it (border turns green background)
3. Go to the destination window
4. `C-a :` then `join-pane` — the marked pane moves here

This is especially useful when you realize an AI tool's output pane belongs in a different window context.

## 2.5 Scripting tmux — Automating Your Environment

One of tmux's greatest strengths is scriptability. Every tmux action is a command that can be called from a shell script. This means you can automate the creation of complex layouts.

### The `send-keys` Command

`send-keys` is the workhorse of tmux scripting. It simulates typing into a pane:

```bash
# Send a command to a pane (C-m = Enter key)
tmux send-keys -t mysession:1.0 "npm run dev" C-m
```

The target format is `session:window.pane`:
- `mysession:1.0` — session "mysession", window 1, pane 0
- `:1` — current session, window 1 (any pane)
- `mysession` — session "mysession" (current window/pane)

### A Complete Startup Script

Here's a script that creates a full development environment:

```bash
#!/bin/bash
# dev-setup.sh — Create a tmux dev environment

SESSION="dev"
PROJECT="$HOME/projects/my-api"

# Kill existing session if it exists
tmux kill-session -t $SESSION 2>/dev/null

# Create session with first window named "code"
tmux new-session -d -s $SESSION -n code -c $PROJECT

# Window 1: code — split into editor + shell
tmux send-keys -t $SESSION:code "vim ." C-m
tmux split-window -h -t $SESSION:code -c $PROJECT
tmux send-keys -t $SESSION:code.1 "git status" C-m

# Window 2: tests
tmux new-window -t $SESSION -n tests -c $PROJECT
tmux send-keys -t $SESSION:tests "npm test -- --watch" C-m

# Window 3: logs
tmux new-window -t $SESSION -n logs -c $PROJECT
tmux send-keys -t $SESSION:logs "tail -f logs/app.log" C-m

# Window 4: shell (clean shell for ad-hoc commands)
tmux new-window -t $SESSION -n shell -c $PROJECT

# Go back to window 1
tmux select-window -t $SESSION:code

# Attach
tmux attach -t $SESSION
```

Save this as `~/bin/dev-setup.sh`, make it executable (`chmod +x`), and run it. In seconds, you have a four-window environment with everything running.

![Figure 13](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20250%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%20%20%3Cdefs%3E%3Cmarker%20id%3D%22arrowhead%22%20markerWidth%3D%2210%22%20markerHeight%3D%227%22%20refX%3D%2210%22%20refY%3D%223.5%22%20orient%3D%22auto%22%3E%3Cpolygon%20points%3D%220%200%2C%2010%203.5%2C%200%207%22%20fill%3D%22%23687078%22%2F%3E%3C%2Fmarker%3E%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2222%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2215%22%20font-weight%3D%22700%22%3EScripted%20Layout%3A%204-Window%20Dev%20Environment%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Windows%20--%3E%0A%20%20%3Crect%20x%3D%2230%22%20y%3D%2245%22%20width%3D%22175%22%20height%3D%22105%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%23527FFF%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%2230%22%20y%3D%2245%22%20width%3D%22175%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%23527FFF%22%2F%3E%0A%20%20%3Crect%20x%3D%2230%22%20y%3D%2261%22%20width%3D%22175%22%20height%3D%2216%22%20fill%3D%22%23527FFF%22%2F%3E%0A%20%20%3Ctext%20x%3D%22117%22%20y%3D%2267%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EWindow%201%3A%20code%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2240%22%20y%3D%2295%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3ELeft%3A%20vim%20.%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2240%22%20y%3D%22113%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3ERight%3A%20git%20status%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22220%22%20y%3D%2245%22%20width%3D%22175%22%20height%3D%22105%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%232EA043%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22220%22%20y%3D%2245%22%20width%3D%22175%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%232EA043%22%2F%3E%0A%20%20%3Crect%20x%3D%22220%22%20y%3D%2261%22%20width%3D%22175%22%20height%3D%2216%22%20fill%3D%22%232EA043%22%2F%3E%0A%20%20%3Ctext%20x%3D%22307%22%20y%3D%2267%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EWindow%202%3A%20tests%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22230%22%20y%3D%2295%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Enpm%20test%20--watch%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22230%22%20y%3D%22113%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%28auto-running%29%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22410%22%20y%3D%2245%22%20width%3D%22175%22%20height%3D%22105%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%23FF9900%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22410%22%20y%3D%2245%22%20width%3D%22175%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%23FF9900%22%2F%3E%0A%20%20%3Crect%20x%3D%22410%22%20y%3D%2261%22%20width%3D%22175%22%20height%3D%2216%22%20fill%3D%22%23FF9900%22%2F%3E%0A%20%20%3Ctext%20x%3D%22497%22%20y%3D%2267%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EWindow%203%3A%20logs%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22420%22%20y%3D%2295%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Etail%20-f%20app.log%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22420%22%20y%3D%22113%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3E%28streaming%29%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%22600%22%20y%3D%2245%22%20width%3D%22175%22%20height%3D%22105%22%20rx%3D%2210%22%20fill%3D%22white%22%20stroke%3D%22%238C4FFF%22%20stroke-width%3D%222%22%20filter%3D%22url%28%23shadow%29%22%2F%3E%0A%20%20%3Crect%20x%3D%22600%22%20y%3D%2245%22%20width%3D%22175%22%20height%3D%2232%22%20rx%3D%2210%22%20fill%3D%22%238C4FFF%22%2F%3E%0A%20%20%3Crect%20x%3D%22600%22%20y%3D%2261%22%20width%3D%22175%22%20height%3D%2216%22%20fill%3D%22%238C4FFF%22%2F%3E%0A%20%20%3Ctext%20x%3D%22687%22%20y%3D%2267%22%20text-anchor%3D%22middle%22%20fill%3D%22white%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EWindow%204%3A%20shell%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22610%22%20y%3D%2295%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3EClean%20shell%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22610%22%20y%3D%22113%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3Efor%20ad-hoc%20work%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Status%20bar%20--%3E%0A%20%20%3Crect%20x%3D%2230%22%20y%3D%22170%22%20width%3D%22740%22%20height%3D%2228%22%20rx%3D%224%22%20fill%3D%22%231E1E1E%22%20stroke%3D%22%23333%22%20stroke-width%3D%221%22%2F%3E%0A%20%20%3Ctext%20x%3D%2245%22%20y%3D%22189%22%20fill%3D%22%23FF9900%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%20font-weight%3D%22700%22%3E%20dev%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2295%22%20y%3D%22189%22%20fill%3D%22%23444%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E%E2%94%82%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22110%22%20y%3D%22189%22%20fill%3D%22%23527FFF%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%20font-weight%3D%22700%22%3E1%3Acode%2A%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22180%22%20y%3D%22189%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E2%3Atests%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22250%22%20y%3D%22189%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E3%3Alogs%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22310%22%20y%3D%22189%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E4%3Ashell%3C%2Ftext%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%22225%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2211%22%3ECreated%20in%20~2%20seconds%20by%20running%3A%20.%2Fdev-setup.sh%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 14: A scripted 4-window development environment*

### AI Tool Startup Script

Here's a script tailored for AI coding workflows:

```bash
#!/bin/bash
# ai-dev.sh — tmux layout for AI-assisted development

SESSION="ai-${1:-project}"
DIR="${2:-$(pwd)}"

tmux kill-session -t $SESSION 2>/dev/null

# Window 1: AI tool (main workspace)
tmux new-session -d -s $SESSION -n ai -c "$DIR"

# Window 2: monitoring — split into tests + git
tmux new-window -t $SESSION -n monitor -c "$DIR"
tmux split-window -v -t $SESSION:monitor -c "$DIR"
tmux send-keys -t $SESSION:monitor.0 "# Run tests here" C-m
tmux send-keys -t $SESSION:monitor.1 "watch -n5 git diff --stat" C-m

# Window 3: shell
tmux new-window -t $SESSION -n shell -c "$DIR"

# Focus on AI window
tmux select-window -t $SESSION:ai

tmux attach -t $SESSION
```

Usage:
```bash
./ai-dev.sh my-api ~/projects/my-api
```

Then in window 1, start your AI tool (`kiro-cli chat`, `claude`, etc.). Window 2 shows tests and git changes. Window 3 is a clean shell.

### The `synchronize-panes` Option

A powerful feature for managing multiple servers: when enabled, everything you type is sent to ALL panes in the window simultaneously.

```bash
# Toggle synchronized panes (from command prompt)
:setw synchronize-panes on

# Or bind it to a key
bind S setw synchronize-panes \; display "Sync #{?synchronize-panes,ON,OFF}"
```

**Use case:** You have 4 panes, each SSH'd into a different server. Enable sync, type `uptime`, and all 4 servers respond at once.

> ⚠️ **Be careful with this!** It sends EVERY keystroke to all panes. A mistyped `rm -rf` goes to all servers simultaneously.

## 2.6 Integration Patterns with AI Coding Tools

Let's get specific about how tmux integrates with the AI tools you'll actually use.

### Pattern 1: Kiro CLI + tmux

Kiro CLI runs in your terminal. With tmux, you get:

| Benefit | How |
|---------|-----|
| Session persistence | Start Kiro in a tmux session, detach, come back later |
| Scrollback | `C-a [` to scroll through Kiro's long responses |
| Side-by-side | Split pane: Kiro on left, file explorer or tests on right |
| Multiple contexts | Window 1: Kiro for project A, Window 2: Kiro for project B |

**Recommended layout:**
```
┌─────────────────────┬──────────────┐
│                     │  tests/build │
│   kiro-cli chat     ├──────────────┤
│                     │  git status  │
└─────────────────────┴──────────────┘
```

### Pattern 2: Fire-and-Forget with Named Sessions

```bash
# Start a Kiro session for a specific task, detached
tmux new-session -d -s kiro-refactor -c ~/projects/my-api       'kiro-cli chat'

# Check on it later
tmux attach -t kiro-refactor

# List all AI sessions
tmux ls | grep kiro
```

### Pattern 3: Watching AI Agents Work (Multi-Pane)

When running multiple AI agents (e.g., Claude Code's sub-agents), tmux lets you watch them all:

```bash
# Create a 4-pane grid to watch 4 agents
tmux new-session -d -s agents
tmux split-window -h
tmux split-window -v
tmux select-pane -t 0
tmux split-window -v
tmux select-layout tiled
```

Each pane can run a different agent or watch a different aspect of the work (tests, logs, git diff).

![Figure 14](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20800%20300%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%0A%20%20%3Ctext%20x%3D%22400%22%20y%3D%2222%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2215%22%20font-weight%3D%22700%22%3EPattern%203%3A%20Multi-Agent%20Monitoring%20Grid%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Terminal%20frame%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%2240%22%20width%3D%22700%22%20height%3D%22230%22%20rx%3D%226%22%20fill%3D%22%231E1E1E%22%20stroke%3D%22%23444%22%20stroke-width%3D%221%22%2F%3E%0A%0A%20%20%3C%21--%204%20panes%20in%20a%20grid%20--%3E%0A%20%20%3C%21--%20Top-left%20--%3E%0A%20%20%3Crect%20x%3D%2255%22%20y%3D%2245%22%20width%3D%22343%22%20height%3D%22108%22%20fill%3D%22%231a1a2e%22%2F%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%2265%22%20fill%3D%22%238C4FFF%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3EAgent%201%3A%20Researcher%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%2282%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EAnalyzing%20API%20patterns...%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%2297%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E%E2%9C%93%20Found%2012%20endpoints%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22112%22%20fill%3D%22%23FF9900%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E%E2%9F%B3%20Mapping%20dependencies...%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Top-right%20--%3E%0A%20%20%3Crect%20x%3D%22402%22%20y%3D%2245%22%20width%3D%22343%22%20height%3D%22108%22%20fill%3D%22%231a1a2e%22%2F%3E%0A%20%20%3Ctext%20x%3D%22417%22%20y%3D%2265%22%20fill%3D%22%23527FFF%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3EAgent%202%3A%20Implementer%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22417%22%20y%3D%2282%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3ERefactoring%20user%20handler...%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22417%22%20y%3D%2297%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E%E2%9C%93%20Updated%20error%20types%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22417%22%20y%3D%22112%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E%E2%9C%93%20Added%20validation%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Dividers%20--%3E%0A%20%20%3Cline%20x1%3D%22400%22%20y1%3D%2245%22%20x2%3D%22400%22%20y2%3D%22245%22%20stroke%3D%22%23527FFF%22%20stroke-width%3D%221%22%2F%3E%0A%20%20%3Cline%20x1%3D%2255%22%20y1%3D%22155%22%20x2%3D%22745%22%20y2%3D%22155%22%20stroke%3D%22%23444%22%20stroke-width%3D%221%22%2F%3E%0A%0A%20%20%3C%21--%20Bottom-left%20--%3E%0A%20%20%3Crect%20x%3D%2255%22%20y%3D%22157%22%20width%3D%22343%22%20height%3D%2286%22%20fill%3D%22%231E1E1E%22%2F%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22177%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3ETest%20Runner%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22194%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EPASS%3A%2047%2F48%20tests%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22211%22%20fill%3D%22%23D13212%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EFAIL%3A%20TestDeleteUser%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%2270%22%20y%3D%22228%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EWatching%20for%20changes...%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Bottom-right%20--%3E%0A%20%20%3Crect%20x%3D%22402%22%20y%3D%22157%22%20width%3D%22343%22%20height%3D%2286%22%20fill%3D%22%231E1E1E%22%2F%3E%0A%20%20%3Ctext%20x%3D%22417%22%20y%3D%22177%22%20fill%3D%22%23FF9900%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%20font-weight%3D%22600%22%3EGit%20Diff%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22417%22%20y%3D%22194%22%20fill%3D%22%23D13212%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EM%20%20handlers%2Fuser.go%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22417%22%20y%3D%22211%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3EA%20%20handlers%2Fuser_test.go%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22417%22%20y%3D%22228%22%20fill%3D%22%23D4D4D4%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%3E%2B89%20-12%20lines%20changed%3C%2Ftext%3E%0A%0A%20%20%3C%21--%20Status%20bar%20--%3E%0A%20%20%3Crect%20x%3D%2250%22%20y%3D%22246%22%20width%3D%22700%22%20height%3D%2224%22%20fill%3D%22%231E1E1E%22%20stroke%3D%22%23333%22%20stroke-width%3D%221%22%2F%3E%0A%20%20%3Ctext%20x%3D%2265%22%20y%3D%22262%22%20fill%3D%22%23FF9900%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%20font-weight%3D%22700%22%3E%20agents%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22130%22%20y%3D%22262%22%20fill%3D%22%23527FFF%22%20font-family%3D%22monospace%22%20font-size%3D%2210%22%20font-weight%3D%22700%22%3E1%3Amonitor%2A%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

*Figure 15: Four-pane grid monitoring multiple AI agents and their effects*

### Pattern 4: The `capture-pane` and `pipe-pane` Commands

These commands let you programmatically capture tmux output — useful for logging AI sessions:

```bash
# Capture the visible content of a pane to a file
tmux capture-pane -t mysession:1.0 -p > ~/ai-output.txt

# Capture the entire scrollback history
tmux capture-pane -t mysession:1.0 -p -S - > ~/full-history.txt

# Pipe all future output from a pane to a file (live logging)
tmux pipe-pane -t mysession:1.0 'cat >> ~/ai-session.log'

# Stop piping
tmux pipe-pane -t mysession:1.0
```

**Use case:** Start an AI coding session, enable `pipe-pane` to log everything, then review the log later to see what the AI did and why.

## 2.7 Part 2 Summary — Key Takeaways

### Configuration

| Setting | Why It Matters |
|---------|---------------|
| `prefix C-a` | Ergonomic prefix key |
| `base-index 1` | Window numbers match keyboard |
| `mouse on` | Scroll, click, resize with mouse |
| `escape-time 0` | No lag in vim |
| `history-limit 50000` | Enough scrollback for AI output |
| `mode-keys vi` | Vim-style navigation in copy mode |
| Split with pipe and `-` | Intuitive split keys |
| `Alt+arrows` for panes | No prefix needed for navigation |

### Copy Mode

| Action | Keys |
|--------|------|
| Enter copy mode | `C-a [` |
| Search forward | `/pattern` |
| Search backward | `?pattern` |
| Start selection | `v` |
| Copy to clipboard | `y` |
| Paste | `C-a ]` |
| Exit | `q` |

### Plugins

| Plugin | Purpose |
|--------|---------|
| TPM | Plugin manager (`C-a I` to install) |
| tmux-resurrect | Save/restore sessions (`C-a Ctrl+s` / `C-a Ctrl+r`) |
| tmux-continuum | Auto-save every 15 minutes |
| tmux-yank | System clipboard integration |

### Scripting

| Command | Purpose |
|---------|---------|
| `tmux new-session -d -s name` | Create detached session |
| `tmux send-keys -t target "cmd" C-m` | Send command to pane |
| `tmux split-window -h/-v` | Split pane in script |
| `tmux capture-pane -p` | Capture pane output |
| `tmux pipe-pane` | Live-log pane output |

### AI Tool Patterns

1. **Fire-and-forget** — detach while AI works
2. **Side-by-side** — AI tool + tests/git in split panes
3. **Multi-agent grid** — watch multiple agents simultaneously
4. **Session logging** — capture-pane / pipe-pane for audit trails

---

### What's Next

**Part 3: Hands-On Lab** will give you progressive exercises to practice everything from Parts 1 and 2:
- Basic navigation drills
- Multi-pane workflow exercises
- Session management scenarios
- Building and testing your `.tmux.conf`
- Real-world scenarios: Kiro CLI + tmux, remote dev, pair programming

---

> 📚 **Sources used in this part:**
> - [tmux GitHub Wiki — Getting Started](https://github.com/tmux/tmux/wiki/Getting-Started) (official documentation)
> - [TPM — Tmux Plugin Manager](https://github.com/tmux-plugins/tpm)
> - Various community configs and tutorials (content was rephrased for compliance with licensing restrictions)

---

# Part 3: Hands-On Lab

> **Target study time: ~2 hours** · These are exercises you do in your terminal alongside this notebook. Read the instructions, then switch to your terminal and do them.

---

## Prerequisites

Before starting, make sure you have:

1. **tmux installed** — run `tmux -V` to verify (should show 3.x)
2. **The starter `.tmux.conf`** from Part 2 installed at `~/.tmux.conf`
3. **TPM installed** — `~/.tmux/plugins/tpm/` should exist
4. A terminal open (iTerm2, Terminal.app, or any terminal emulator)

If you haven't set up the config yet, do it now:

```bash
# Quick setup (copy the config from Part 2)
# Then install TPM:
git clone https://github.com/tmux-plugins/tpm ~/.tmux/plugins/tpm
```

> **Convention in this lab:**
> - 📋 = instruction (do this in your terminal)
> - ✅ = expected result (what you should see)
> - 💡 = tip or explanation
> - 🏆 = challenge (try it yourself before reading the answer)

---

## Lab 1: Session Basics (15 minutes)

### Exercise 1.1: Create, Detach, Reattach

📋 **Step 1:** Create a named session
```bash
tmux new -s lab
```

✅ You should see a status bar at the bottom showing `[lab] 1:zsh*` (or `1:bash*`).

📋 **Step 2:** Run a command to prove you're in tmux
```bash
echo "I am in tmux session: $TMUX"
```

✅ You should see a long string containing the tmux socket path. If `$TMUX` is empty, you're not inside tmux.

📋 **Step 3:** Detach
Press `C-a d` (Ctrl+a, then d)

✅ You should see:
```
[detached (from session lab)]
```
You're back in your regular shell.

📋 **Step 4:** List sessions
```bash
tmux ls
```

✅ Output:
```
lab: 1 windows (created Tue Apr 21 15:30:00 2026)
```

📋 **Step 5:** Reattach
```bash
tmux attach -t lab
```

✅ You're back in the same session. Your `echo` output is still visible in the scrollback.

💡 **Key insight:** The session was alive the entire time you were detached. If this were a remote server and your SSH dropped, the same thing would happen — the session survives.

### Exercise 1.2: Multiple Sessions

📋 **Step 1:** Detach from the lab session
Press `C-a d`

📋 **Step 2:** Create a second session
```bash
tmux new -s work
```

📋 **Step 3:** Create a third session
Press `C-a d` to detach, then:
```bash
tmux new -s personal
```

📋 **Step 4:** List all sessions
Press `C-a d` to detach, then:
```bash
tmux ls
```

✅ Output (order may vary):
```
lab: 1 windows (created ...)
personal: 1 windows (created ...)
work: 1 windows (created ...)
```

📋 **Step 5:** Attach to a specific session
```bash
tmux attach -t work
```

📋 **Step 6:** Switch sessions from inside tmux
Press `C-a s` to open the session picker (tree mode).

✅ You should see all three sessions listed. Use arrow keys to navigate, `Enter` to switch.

📋 **Step 7:** Clean up — kill all sessions
Press `C-a d` to detach, then:
```bash
tmux kill-server
```

✅ `tmux ls` should now show `no server running`.

💡 `kill-server` is the nuclear option — it kills ALL sessions. Use `kill-session -t name` to kill just one.

## Lab 2: Windows (15 minutes)

### Exercise 2.1: Creating and Navigating Windows

📋 **Step 1:** Start a fresh session
```bash
tmux new -s windows-lab
```

📋 **Step 2:** Create 3 more windows
- Press `C-a c` — new window (you're now in window 2)
- Press `C-a c` — new window (window 3)
- Press `C-a c` — new window (window 4)

✅ Status bar should show: `1:zsh  2:zsh  3:zsh  4:zsh*`

📋 **Step 3:** Navigate by number
- Press `C-a 1` — go to window 1
- Press `C-a 3` — go to window 3
- Press `C-a 4` — go to window 4

✅ The `*` in the status bar moves to the window you select.

📋 **Step 4:** Navigate sequentially
- Press `Shift+Left` — previous window
- Press `Shift+Right` — next window

💡 These work without the prefix because of our `-n` bindings in `.tmux.conf`.

📋 **Step 5:** Toggle last window
- Press `C-a l` (lowercase L)

✅ You jump to the window you were just in. Press again to toggle back. This is like `Alt+Tab` for tmux windows.

### Exercise 2.2: Naming and Organizing Windows

📋 **Step 1:** Go to window 1 (`C-a 1`)

📋 **Step 2:** Rename it
Press `C-a ,` — a prompt appears at the bottom. Type `editor` and press `Enter`.

✅ Status bar now shows `1:editor*` instead of `1:zsh*`.

📋 **Step 3:** Rename the others
- `C-a 2`, then `C-a ,` → type `tests`
- `C-a 3`, then `C-a ,` → type `logs`
- `C-a 4`, then `C-a ,` → type `shell`

✅ Status bar: `1:editor  2:tests  3:logs  4:shell*`

📋 **Step 4:** Run something in each window
- Window 1 (`C-a 1`): `echo "This is the editor window"`
- Window 2 (`C-a 2`): `echo "This is the tests window"`
- Window 3 (`C-a 3`): `echo "This is the logs window"`

📋 **Step 5:** Use the window picker
Press `C-a w`

✅ Tree mode opens showing all windows with previews. You can see the `echo` output in each window's preview pane.

📋 **Step 6:** Kill a window
Go to window 4 (`C-a 4`), then press `C-a &`. Type `y` to confirm.

✅ Window 4 is gone. With `renumber-windows on`, the remaining windows stay as 1, 2, 3 (no gaps).

🏆 **Challenge:** Without looking at the instructions, create a new window, name it "monitoring", and move it to be window 1 (hint: `C-a .` to move, enter the target index).

## Lab 3: Panes (20 minutes)

### Exercise 3.1: Splitting and Navigating

📋 **Step 1:** Go to window 1 (or create a fresh session: `tmux new -s panes-lab`)

📋 **Step 2:** Split vertically (left/right)
Press `C-a |`

✅ Your window is now split into two panes side by side. The cursor is in the right pane.

![Figure 15](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20600%20160%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%20%20%3Ctext%20x%3D%22300%22%20y%3D%2218%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22700%22%3EAfter%20C-a%20%7C%20%28vertical%20split%29%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%2230%22%20y%3D%2230%22%20width%3D%22540%22%20height%3D%22100%22%20rx%3D%226%22%20fill%3D%22%231E1E1E%22%20stroke%3D%22%23444%22%20stroke-width%3D%221%22%2F%3E%0A%20%20%3Crect%20x%3D%2235%22%20y%3D%2235%22%20width%3D%22262%22%20height%3D%2290%22%20fill%3D%22%231a1a2e%22%2F%3E%0A%20%20%3Ctext%20x%3D%22166%22%20y%3D%2280%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%3EPane%200%20%28original%29%3C%2Ftext%3E%0A%20%20%3Cline%20x1%3D%22300%22%20y1%3D%2235%22%20x2%3D%22300%22%20y2%3D%22125%22%20stroke%3D%22%23444%22%20stroke-width%3D%221%22%2F%3E%0A%20%20%3Crect%20x%3D%22303%22%20y%3D%2235%22%20width%3D%22262%22%20height%3D%2290%22%20fill%3D%22%231a1a2e%22%2F%3E%0A%20%20%3Ctext%20x%3D%22434%22%20y%3D%2275%22%20text-anchor%3D%22middle%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2212%22%20font-weight%3D%22600%22%3EPane%201%20%28new%2C%20active%29%3C%2Ftext%3E%0A%20%20%3Ctext%20x%3D%22434%22%20y%3D%2292%22%20text-anchor%3D%22middle%22%20fill%3D%22%234EC9B0%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3E%24%20%E2%96%8A%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

📋 **Step 3:** Move to the left pane
Press `Alt+Left` (or `C-a Left`)

✅ The cursor moves to the left pane. The active pane border changes.

📋 **Step 4:** Split the left pane horizontally (top/bottom)
Press `C-a -`

✅ The left side is now split into top and bottom panes. You have 3 panes total.

![Figure 16](data:image/svg+xml,%3Csvg%20xmlns%3D%22http%3A%2F%2Fwww.w3.org%2F2000%2Fsvg%22%20viewBox%3D%220%200%20600%20160%22%3E%0A%20%20%3Cdefs%3E%0A%20%20%3Cfilter%20id%3D%22shadow%22%20x%3D%22-5%25%22%20y%3D%22-5%25%22%20width%3D%22115%25%22%20height%3D%22115%25%22%3E%0A%20%20%20%20%3CfeDropShadow%20dx%3D%222%22%20dy%3D%222%22%20stdDeviation%3D%223%22%20flood-opacity%3D%220.15%22%2F%3E%0A%20%20%3C%2Ffilter%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_blue%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23527FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_purple%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%238C4FFF%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%20%20%3ClinearGradient%20id%3D%22grad_orange%22%20x1%3D%220%25%22%20y1%3D%220%25%22%20x2%3D%220%25%22%20y2%3D%22100%25%22%3E%0A%20%20%20%20%3Cstop%20offset%3D%220%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.15%22%2F%3E%0A%20%20%20%20%3Cstop%20offset%3D%22100%25%22%20style%3D%22stop-color%3A%23FF9900%3Bstop-opacity%3A0.05%22%2F%3E%0A%20%20%3C%2FlinearGradient%3E%0A%3C%2Fdefs%3E%0A%20%20%3Ctext%20x%3D%22300%22%20y%3D%2218%22%20text-anchor%3D%22middle%22%20fill%3D%22%23232F3E%22%20font-family%3D%22system-ui%2Csans-serif%22%20font-size%3D%2213%22%20font-weight%3D%22700%22%3EAfter%20C-a%20-%20%28horizontal%20split%20on%20left%20pane%29%3C%2Ftext%3E%0A%20%20%3Crect%20x%3D%2230%22%20y%3D%2230%22%20width%3D%22540%22%20height%3D%22110%22%20rx%3D%226%22%20fill%3D%22%231E1E1E%22%20stroke%3D%22%23444%22%20stroke-width%3D%221%22%2F%3E%0A%20%20%3C%21--%20Top-left%20--%3E%0A%20%20%3Crect%20x%3D%2235%22%20y%3D%2235%22%20width%3D%22262%22%20height%3D%2245%22%20fill%3D%22%231a1a2e%22%2F%3E%0A%20%20%3Ctext%20x%3D%22166%22%20y%3D%2262%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3EPane%200%20%28top-left%29%3C%2Ftext%3E%0A%20%20%3C%21--%20Bottom-left%20--%3E%0A%20%20%3Cline%20x1%3D%2235%22%20y1%3D%2282%22%20x2%3D%22297%22%20y2%3D%2282%22%20stroke%3D%22%23444%22%20stroke-width%3D%221%22%2F%3E%0A%20%20%3Crect%20x%3D%2235%22%20y%3D%2284%22%20width%3D%22262%22%20height%3D%2251%22%20fill%3D%22%231a1a2e%22%2F%3E%0A%20%20%3Ctext%20x%3D%22166%22%20y%3D%22112%22%20text-anchor%3D%22middle%22%20fill%3D%22%232EA043%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%20font-weight%3D%22600%22%3EPane%202%20%28bottom-left%2C%20active%29%3C%2Ftext%3E%0A%20%20%3C%21--%20Right%20--%3E%0A%20%20%3Cline%20x1%3D%22300%22%20y1%3D%2235%22%20x2%3D%22300%22%20y2%3D%22135%22%20stroke%3D%22%23444%22%20stroke-width%3D%221%22%2F%3E%0A%20%20%3Crect%20x%3D%22303%22%20y%3D%2235%22%20width%3D%22262%22%20height%3D%22100%22%20fill%3D%22%231a1a2e%22%2F%3E%0A%20%20%3Ctext%20x%3D%22434%22%20y%3D%2285%22%20text-anchor%3D%22middle%22%20fill%3D%22%23687078%22%20font-family%3D%22monospace%22%20font-size%3D%2211%22%3EPane%201%20%28right%29%3C%2Ftext%3E%0A%3C%2Fsvg%3E)

📋 **Step 5:** Run a different command in each pane

Navigate to each pane with `Alt+arrows` and run:
- Pane 0 (top-left): `echo "I am pane 0" && sleep 999`
- Pane 1 (right): `echo "I am pane 1" && top`
- Pane 2 (bottom-left): `echo "I am pane 2"`

📋 **Step 6:** Show pane numbers
Press `C-a q`

✅ Large numbers appear on each pane for 3 seconds. Press a number to jump to that pane.

📋 **Step 7:** Zoom a pane
Navigate to the right pane (showing `top`), then press `C-a z`.

✅ The right pane expands to fill the entire window. The status bar shows a `Z` flag. Press `C-a z` again to unzoom.

💡 **Zoom is your best friend.** When an AI tool produces a long response, zoom that pane to read it comfortably, then unzoom to see your other panes.

### Exercise 3.2: Resizing Panes

📋 **Step 1:** Make sure you have the 3-pane layout from Exercise 3.1

📋 **Step 2:** Resize the right pane wider
Navigate to the right pane, then press `C-a Shift+Right` several times.

✅ The right pane grows wider with each press. The `-r` flag in our config makes this repeatable — you only need to press the prefix once, then keep pressing `Shift+Right`.

📋 **Step 3:** Try the preset layouts
Press `C-a Space` repeatedly to cycle through layouts:

| Layout | Description |
|--------|-------------|
| even-horizontal | All panes side by side, equal width |
| even-vertical | All panes stacked, equal height |
| main-horizontal | One big pane on top, rest below |
| main-vertical | One big pane on left, rest on right |
| tiled | Grid layout |

✅ Each press of `C-a Space` switches to the next layout. Watch how the panes rearrange.

💡 `main-vertical` is the most popular layout for AI coding: big pane on the left for the AI tool, smaller panes on the right for tests/git.

### Exercise 3.3: Breaking and Joining Panes

📋 **Step 1:** Navigate to the bottom-left pane

📋 **Step 2:** Break it into its own window
Press `C-a !`

✅ The pane becomes a new window. Check the status bar — you now have an additional window.

📋 **Step 3:** Join it back
Press `C-a :` to open the command prompt, then type:
```
join-pane -s :2
```
(Replace `:2` with the window number of the pane you just broke out)

✅ The pane is back in the original window.

🏆 **Challenge:** Create a 4-pane grid layout (2×2) using only split commands and `C-a Space` to get the tiled layout. Then zoom one pane, read its content, and unzoom.

## Lab 4: Copy Mode & Scrollback (15 minutes)

### Exercise 4.1: Scrolling Through Output

📋 **Step 1:** Generate a lot of output
```bash
for i in $(seq 1 200); do echo "Line $i: $(date) - The quick brown fox jumps over the lazy dog"; done
```

📋 **Step 2:** Enter copy mode
Press `C-a [`

✅ The pane freezes. A line counter appears in the top-right corner (e.g., `[0/200]`).

📋 **Step 3:** Navigate
- Press `k` or `Up` to scroll up one line
- Press `j` or `Down` to scroll down one line
- Press `Ctrl+u` to scroll up half a page
- Press `Ctrl+d` to scroll down half a page
- Press `g` to jump to the very top
- Press `G` to jump to the very bottom

📋 **Step 4:** Exit copy mode
Press `q`

✅ The pane unfreezes and shows the latest output again.

💡 **Mouse scrolling:** With `mouse on` in your config, you can also scroll with your trackpad. This automatically enters copy mode.

### Exercise 4.2: Searching

📋 **Step 1:** Generate output with specific patterns
```bash
for i in $(seq 1 100); do
  if [ $((i % 10)) -eq 0 ]; then
    echo "Line $i: ERROR - Something went wrong!"
  else
    echo "Line $i: OK - All systems normal"
  fi
done
```

📋 **Step 2:** Enter copy mode and search
- Press `C-a [`
- Press `/` (search forward)
- Type `ERROR` and press `Enter`

✅ The cursor jumps to the first match of "ERROR". The match is highlighted.

📋 **Step 3:** Jump between matches
- Press `n` — next match
- Press `N` — previous match

✅ You can cycle through all 10 ERROR lines.

📋 **Step 4:** Search backward
- Press `?`
- Type `Line 50` and press `Enter`

✅ The cursor jumps backward to "Line 50".

📋 **Step 5:** Exit
Press `q`

### Exercise 4.3: Copying and Pasting

📋 **Step 1:** Generate some output
```bash
echo "Name: John Doe"
echo "Email: john@example.com"
echo "Role: Developer"
echo "Team: Platform"
```

📋 **Step 2:** Enter copy mode
Press `C-a [`

📋 **Step 3:** Navigate to "Email:" line
Use `k`/`j` or search with `/Email`

📋 **Step 4:** Start selection
Press `v` (with vi mode)

✅ Selection starts at the cursor position. As you move, text is highlighted.

📋 **Step 5:** Select the email line
Press `$` to select to end of line (or use `l` to move right)

📋 **Step 6:** Copy
Press `y`

✅ The selection is copied and you exit copy mode. With our config, it's also in your system clipboard.

📋 **Step 7:** Paste in tmux
Press `C-a ]`

✅ The copied text is pasted at the cursor.

📋 **Step 8:** Paste in another app
Open any text editor or browser and `Cmd+V` (macOS).

✅ The same text pastes from the system clipboard (thanks to `pbcopy` in our config).

🏆 **Challenge:** Generate 500 lines of output, search for a specific line, select 3 lines around it, copy them, and paste into a different pane.

## Lab 5: Configuration & Plugins (15 minutes)

### Exercise 5.1: Testing Your Config

📋 **Step 1:** Verify your config is loaded
Start tmux and check the prefix key:
```bash
tmux show -g prefix
```

✅ Should show `prefix C-a` (if you installed the Part 2 config).

📋 **Step 2:** Test the reload keybinding
Open `~/.tmux.conf` in a pane and add this line at the end:
```bash
set -g status-style 'bg=#2E1065,fg=#D4D4D4'
```

Now press `C-a r`

✅ You should see "Config reloaded!" in the status bar, and the bar should turn purple.

📋 **Step 3:** Revert the change
Remove the line you just added (or change it back to `bg=#1E1E1E`), then `C-a r` again.

💡 This is the workflow for config experimentation: edit → reload → see the result instantly.

### Exercise 5.2: Installing Plugins with TPM

📋 **Step 1:** Verify TPM is installed
```bash
ls ~/.tmux/plugins/tpm/
```

✅ You should see TPM's files (README.md, tpm, etc.)

📋 **Step 2:** Make sure your `.tmux.conf` has the plugin block
The bottom of your config should have:
```bash
set -g @plugin 'tmux-plugins/tpm'
set -g @plugin 'tmux-plugins/tmux-resurrect'
set -g @plugin 'tmux-plugins/tmux-yank'

run '~/.tmux/plugins/tpm/tpm'
```

📋 **Step 3:** Install plugins
Press `C-a I` (capital I)

✅ TPM clones the plugins and shows a success message. This takes a few seconds.

📋 **Step 4:** Test tmux-resurrect
- Create a specific layout: 2 windows, one with split panes
- Press `C-a Ctrl+s` to save

✅ You should see "Tmux environment saved!" in the status bar.

📋 **Step 5:** Kill and restore
```bash
tmux kill-server
tmux new -s test
```
Press `C-a Ctrl+r` to restore.

✅ Your windows and pane layout are restored! The programs that were running may need to be restarted, but the layout is back.

💡 With `tmux-continuum`, this save happens automatically every 15 minutes. You never lose your layout.

## Lab 6: Scripting tmux (15 minutes)

### Exercise 6.1: Your First tmux Script

📋 **Step 1:** Create a script file
```bash
cat > ~/bin/tmux-lab.sh << 'EOF'
#!/bin/bash
SESSION="lab-scripted"

tmux kill-session -t $SESSION 2>/dev/null

# Window 1: main workspace with 2 panes
tmux new-session -d -s $SESSION -n main
tmux split-window -h -t $SESSION:main
tmux send-keys -t $SESSION:main.0 "echo 'Left pane ready'" C-m
tmux send-keys -t $SESSION:main.1 "echo 'Right pane ready'" C-m

# Window 2: monitoring
tmux new-window -t $SESSION -n monitor
tmux send-keys -t $SESSION:monitor "echo 'Monitor window ready'" C-m

# Select first window
tmux select-window -t $SESSION:main
tmux select-pane -t $SESSION:main.0

# Attach
tmux attach -t $SESSION
EOF
mkdir -p ~/bin
chmod +x ~/bin/tmux-lab.sh
```

📋 **Step 2:** Run it
```bash
~/bin/tmux-lab.sh
```

✅ You should see:
- Window "main" with 2 panes, each showing their echo message
- Window "monitor" (switch with `Shift+Right`)
- Cursor in the left pane of "main"

📋 **Step 3:** Detach and run again
Press `C-a d`, then run `~/bin/tmux-lab.sh` again.

✅ It kills the old session and creates a fresh one. This is idempotent — safe to run repeatedly.

### Exercise 6.2: AI Development Layout Script

📋 **Step 1:** Create the script
```bash
cat > ~/bin/ai-workspace.sh << 'EOF'
#!/bin/bash
# Usage: ai-workspace.sh [project-name] [directory]
SESSION="ai-${1:-default}"
DIR="${2:-$(pwd)}"

tmux kill-session -t $SESSION 2>/dev/null

# Window 1: AI tool (full width for the AI agent)
tmux new-session -d -s $SESSION -n ai -c "$DIR"
tmux send-keys -t $SESSION:ai "echo '🤖 Start your AI tool here: kiro-cli chat'" C-m

# Window 2: code + tests side by side
tmux new-window -t $SESSION -n dev -c "$DIR"
tmux split-window -h -t $SESSION:dev -c "$DIR"
tmux send-keys -t $SESSION:dev.0 "echo '📝 Editor pane'" C-m
tmux send-keys -t $SESSION:dev.1 "echo '🧪 Test runner pane'" C-m

# Window 3: git + logs
tmux new-window -t $SESSION -n ops -c "$DIR"
tmux split-window -v -t $SESSION:ops -c "$DIR"
tmux send-keys -t $SESSION:ops.0 "echo '📊 Git status pane'" C-m
tmux send-keys -t $SESSION:ops.1 "echo '📋 Logs pane'" C-m

# Focus on AI window
tmux select-window -t $SESSION:ai

tmux attach -t $SESSION
EOF
chmod +x ~/bin/ai-workspace.sh
```

📋 **Step 2:** Run it
```bash
~/bin/ai-workspace.sh my-project ~/Downloads
```

✅ Three windows:
- `ai` — full-width pane for your AI tool
- `dev` — split into editor + tests
- `ops` — split into git + logs

📋 **Step 3:** Explore
- `Shift+Right` / `Shift+Left` to switch windows
- `Alt+arrows` to switch panes within a window
- `C-a z` to zoom any pane

🏆 **Challenge:** Modify the script to add a 4th window called "shell" that opens in the project directory. Then add `watch -n5 'date'` to the logs pane instead of the echo.

## Lab 7: Real-World Scenarios (20 minutes)

These exercises simulate actual workflows you'll encounter.

### Scenario 7.1: The SSH Drop Recovery

This simulates what happens when your connection drops while working remotely.

📋 **Step 1:** Create a session and start a "long-running task"
```bash
tmux new -s remote-work
```
In the session, run:
```bash
# Simulate a long-running process
for i in $(seq 1 60); do echo "Processing batch $i/60..."; sleep 1; done; echo "DONE!"
```

📋 **Step 2:** Simulate a connection drop
While the process is running (don't wait for it to finish), press `C-a d` to detach.

📋 **Step 3:** Wait 10 seconds, then "reconnect"
```bash
sleep 10
tmux attach -t remote-work
```

✅ The process continued running while you were detached! You can see the output it produced while you were away. If it finished, you'll see "DONE!".

💡 **This is exactly what happens with SSH drops.** The tmux session on the server keeps running. When you SSH back in and reattach, everything is there.

### Scenario 7.2: Multi-Project Context Switching

You're a TAM working on multiple customer projects. Each needs its own context.

📋 **Step 1:** Create project sessions
```bash
tmux new -d -s customer-alpha -c ~/Downloads
tmux new -d -s customer-beta -c ~/Downloads
tmux new -d -s internal-work -c ~/Downloads
```

📋 **Step 2:** Set up each session
```bash
# Customer Alpha: 2 windows
tmux send-keys -t customer-alpha "echo 'Alpha: reviewing architecture'" C-m
tmux new-window -t customer-alpha -n docs
tmux send-keys -t customer-alpha:docs "echo 'Alpha: documentation'" C-m

# Customer Beta: 1 window with split
tmux split-window -h -t customer-beta
tmux send-keys -t customer-beta:1.0 "echo 'Beta: cost analysis'" C-m
tmux send-keys -t customer-beta:1.1 "echo 'Beta: RI recommendations'" C-m

# Internal: simple shell
tmux send-keys -t internal-work "echo 'Internal: TFC prep'" C-m
```

📋 **Step 3:** Attach and switch between them
```bash
tmux attach -t customer-alpha
```

Now use `C-a s` (session picker) to switch between all three sessions.

✅ Each session has its own windows, panes, and context. Switching is instant — no need to close anything or lose state.

📋 **Step 4:** Quick-switch with `C-a (` and `C-a )`
These cycle through sessions without opening the picker.

💡 **This is the "virtual desktop" pattern.** Each customer/project gets its own session. You never mix contexts, and you can jump between them in under a second.

### Scenario 7.3: Kiro CLI + tmux Workflow

📋 **Step 1:** Create an AI workspace
```bash
tmux new -s kiro-session
```

📋 **Step 2:** Set up the layout
- Split vertically: `C-a |`
- In the right pane, split horizontally: `C-a -`

You now have:
```
┌──────────────┬──────────┐
│              │ right-top│
│   left       ├──────────┤
│   (AI tool)  │ right-bot│
└──────────────┴──────────┘
```

📋 **Step 3:** Set up each pane
- Left pane (`Alt+Left`): This is where you'd run `kiro-cli chat`
  ```bash
  echo "🤖 AI tool pane — run: kiro-cli chat"
  ```
- Right-top (`Alt+Right`, `Alt+Up`):
  ```bash
  echo "🧪 Test watcher — run: watch -n5 'ls -la'"
  watch -n5 'date'
  ```
- Right-bottom (`Alt+Down`):
  ```bash
  echo "📊 Git monitor"
  watch -n10 'echo "=== Git Status ===" && git status --short 2>/dev/null || echo "Not a git repo"'
  ```

📋 **Step 4:** Practice the workflow
- Focus on the left pane (where the AI tool would be): `Alt+Left`
- Zoom it for reading: `C-a z`
- Unzoom to see all panes: `C-a z`
- Detach: `C-a d`
- Reattach: `tmux attach -t kiro-session`

✅ Everything is exactly as you left it — the `watch` commands are still running, the layout is preserved.

💡 **This is the daily workflow.** Start your AI workspace in the morning, detach when you go to lunch, reattach when you're back. The AI tool's context and your monitoring panes are always there.

## Lab 8: Graduation Challenges (15 minutes)

These challenges combine everything you've learned. Try each one without looking at hints first.

### 🏆 Challenge 1: The Full Setup

**Goal:** Create a complete development environment from scratch.

Requirements:
1. Create a session named "fullstack"
2. Window 1 named "frontend": 2 vertical panes
3. Window 2 named "backend": 1 pane running `top`
4. Window 3 named "db": 2 horizontal panes
5. Rename the session to "my-project"
6. Detach, list sessions, reattach

<details>
<summary>💡 Hint (click to reveal)</summary>

```bash
tmux new -s fullstack -n frontend
# C-a | to split
# C-a c to new window, C-a , to rename to "backend"
# run top
# C-a c, C-a , to rename to "db"
# C-a - to split horizontally
# C-a $ to rename session to "my-project"
# C-a d to detach
# tmux ls
# tmux attach -t my-project
```
</details>

### 🏆 Challenge 2: The Script Builder

**Goal:** Write a shell script that creates this layout automatically:

```
Session: "monitor"
├── Window 1: "dashboard" (3 panes in tiled layout)
│   ├── Pane 0: running 'top'
│   ├── Pane 1: running 'watch -n2 df -h'
│   └── Pane 2: running 'watch -n5 who'
└── Window 2: "shell" (single pane, clean shell)
```

<details>
<summary>💡 Solution (click to reveal)</summary>

```bash
#!/bin/bash
SESSION="monitor"
tmux kill-session -t $SESSION 2>/dev/null

tmux new-session -d -s $SESSION -n dashboard
tmux send-keys -t $SESSION:dashboard "top" C-m

tmux split-window -h -t $SESSION:dashboard
tmux send-keys -t $SESSION:dashboard.1 "watch -n2 df -h" C-m

tmux split-window -v -t $SESSION:dashboard.1
tmux send-keys -t $SESSION:dashboard.2 "watch -n5 who" C-m

tmux select-layout -t $SESSION:dashboard tiled

tmux new-window -t $SESSION -n shell
tmux select-window -t $SESSION:dashboard

tmux attach -t $SESSION
```
</details>

### 🏆 Challenge 3: Copy Mode Mastery

**Goal:** Extract specific information from a large output.

📋 **Step 1:** Generate a fake log file:
```bash
for i in $(seq 1 500); do
  ts=$(date -v-${i}M "+%Y-%m-%d %H:%M:%S" 2>/dev/null || date "+%Y-%m-%d %H:%M:%S")
  level=$( [ $((RANDOM % 10)) -eq 0 ] && echo "ERROR" || echo "INFO" )
  echo "$ts [$level] Request processed: user_$((RANDOM % 100)) action=page_view status=$( [ "$level" = "ERROR" ] && echo "500" || echo "200" )"
done
```

📋 **Step 2:** Using only copy mode (no grep!):
1. Find the first ERROR line
2. Copy the entire ERROR line to your clipboard
3. Count how many ERROR lines there are (hint: search and count `n` presses)
4. Find the last ERROR line and copy the timestamp

<details>
<summary>💡 Approach</summary>

1. `C-a [` → `/ERROR` → `Enter` (finds first match)
2. `0` (start of line) → `v` → `$` (end of line) → `y` (copy)
3. `C-a [` → `/ERROR` → `Enter` → keep pressing `n` and count
4. `C-a [` → `?ERROR` → `Enter` (search backward from bottom) → `0` → `v` → `f ` (find space) → `y`
</details>

### 🏆 Challenge 4: The Context Juggler

**Goal:** Simulate a TAM's daily workflow.

1. Create 3 sessions: "customer-A", "customer-B", "admin"
2. In "customer-A": create windows "review" and "notes"
3. In "customer-B": create a 2-pane layout
4. In "admin": just a shell
5. Practice switching between all sessions using `C-a s`
6. From "customer-A", quickly check something in "admin" and come back
7. Kill only "admin" without affecting the other sessions
8. Verify the other sessions are still alive

💡 **Hint for step 7:** `tmux kill-session -t admin` from the command prompt, or `x` in tree mode.

---

## 🎓 Lab Complete!

If you completed all 8 labs, you now have practical muscle memory for:

| Skill | Labs |
|-------|------|
| Session lifecycle (create, detach, attach, kill) | Lab 1 |
| Window management (create, name, navigate, kill) | Lab 2 |
| Pane management (split, navigate, resize, zoom) | Lab 3 |
| Copy mode (scroll, search, select, copy, paste) | Lab 4 |
| Configuration and plugins | Lab 5 |
| Scripting tmux | Lab 6 |
| Real-world workflows | Lab 7 |
| Combined skills | Lab 8 |

**What's next:** Part 4 (Appendix) has a complete cheat sheet you can print or keep open as a reference while you build muscle memory.

> 📚 **Practice tip:** Use tmux for ALL your terminal work for the next week. Even if it feels slower at first, the muscle memory builds fast. After 3-4 days, you won't want to use a terminal without it.

---

# Part 4: Appendix & Cheat Sheet

> **Reference material** · Print this, bookmark it, or keep it open in a pane while you build muscle memory.

---

## 4.1 Complete Keybinding Cheat Sheet

All keybindings below assume the **Part 2 config** (prefix = `C-a`). If you use the default prefix, substitute `C-b`.

### Sessions

| Keys / Command | Action |
|----------------|--------|
| `tmux new -s name` | Create named session |
| `tmux attach -t name` | Attach to session |
| `tmux new -As name` | Attach or create |
| `tmux ls` | List sessions |
| `tmux kill-session -t name` | Kill session |
| `tmux kill-server` | Kill everything |
| `C-a d` | Detach |
| `C-a s` | Session picker (tree mode) |
| `C-a $` | Rename session |
| `C-a (` | Previous session |
| `C-a )` | Next session |

### Windows

| Keys | Action |
|------|--------|
| `C-a c` | New window (in current directory) |
| `C-a ,` | Rename window |
| `C-a &` | Kill window (with confirmation) |
| `C-a w` | Window picker (tree mode) |
| `C-a 1-9` | Go to window by number |
| `C-a l` | Toggle last window |
| `Shift+Left` | Previous window (no prefix) |
| `Shift+Right` | Next window (no prefix) |
| `C-a .` | Move window to new index |

### Panes

| Keys | Action |
|------|--------|
| `C-a \|` | Split vertically (left/right) |
| `C-a -` | Split horizontally (top/bottom) |
| `Alt+←↑↓→` | Move between panes (no prefix) |
| `C-a z` | Zoom/unzoom pane |
| `C-a x` | Kill pane |
| `C-a !` | Break pane into own window |
| `C-a q` | Show pane numbers |
| `C-a {` | Swap with previous pane |
| `C-a }` | Swap with next pane |
| `C-a Space` | Cycle layouts |
| `C-a Shift+←→↑↓` | Resize pane |

### Copy Mode (vi keys)

| Keys | Action |
|------|--------|
| `C-a [` | Enter copy mode |
| `q` | Exit copy mode |
| `h j k l` | Move cursor |
| `C-u / C-d` | Half-page up/down |
| `g / G` | Top / bottom of history |
| `/pattern` | Search forward |
| `?pattern` | Search backward |
| `n / N` | Next / previous match |
| `v` | Start selection |
| `y` | Copy (to clipboard) |
| `C-a ]` | Paste |

### Other

| Keys | Action |
|------|--------|
| `C-a :` | Command prompt |
| `C-a ?` | List all keybindings |
| `C-a r` | Reload config |
| `C-a I` | Install TPM plugins |
| `C-a Ctrl+s` | Save session (resurrect) |
| `C-a Ctrl+r` | Restore session (resurrect) |

## 4.2 Troubleshooting Common Issues

### "My prefix key doesn't work"

**Symptom:** Pressing `C-a` (or `C-b`) does nothing.

**Causes & fixes:**
1. **Not inside tmux** — check with `echo $TMUX` (should be non-empty)
2. **Config not loaded** — run `tmux source ~/.tmux.conf` from the command prompt
3. **Typo in config** — check for errors: `tmux source ~/.tmux.conf` will show parse errors
4. **Terminal intercepting the key** — some terminals (especially on macOS) capture `C-a` for "select all". Check your terminal's keybinding settings.

### "Colors look wrong"

**Symptom:** Colors in vim/neovim are washed out or wrong inside tmux.

**Fix:** Make sure your config has:
```bash
set -g default-terminal "tmux-256color"
set -ag terminal-overrides ",xterm-256color:RGB"
```
Then start a **new** tmux session (these settings don't apply to existing sessions).

### "Mouse scrolling doesn't work"

**Symptom:** Trackpad/mouse wheel doesn't scroll in tmux.

**Fix:** Add to config:
```bash
set -g mouse on
```
Reload with `C-a r`. Mouse scrolling automatically enters copy mode.

### "Escape key is slow in vim"

**Symptom:** Pressing `Escape` in vim has a noticeable delay.

**Fix:**
```bash
set -sg escape-time 0
```
tmux waits after `Escape` to see if it's part of a longer sequence. Setting to 0 eliminates the delay.

### "Copy doesn't go to system clipboard"

**Symptom:** Copying in tmux copy mode doesn't paste in other apps.

**Fix for macOS:**
```bash
bind -T copy-mode-vi y send -X copy-pipe-and-cancel "pbcopy"
```

**Fix for Linux (X11):**
```bash
bind -T copy-mode-vi y send -X copy-pipe-and-cancel "xclip -selection clipboard"
```

**Fix for Linux (Wayland):**
```bash
bind -T copy-mode-vi y send -X copy-pipe-and-cancel "wl-copy"
```

### "tmux-resurrect doesn't restore my programs"

**Symptom:** After restore, panes are empty shells instead of running programs.

**Explanation:** By default, resurrect only restores a limited set of programs (vim, ssh, etc.). Add to config:
```bash
set -g @resurrect-processes 'vim nvim ssh top htop "~kiro-cli"'
```
The `~` prefix means "restore with the original arguments."

### "Nested tmux (tmux inside tmux)"

**Symptom:** You SSH into a server that also runs tmux. Both use the same prefix key.

**Fix:** Press the prefix **twice** to send it to the inner tmux:
- `C-a C-a <command>` — sends the command to the inner tmux
- `C-a <command>` — sends to the outer tmux

Or use a different prefix for the inner tmux in the remote server's config.

## 4.3 The Complete `.tmux.conf` (Final Version)

This is the full config from Part 2, plus plugin settings. Copy this into `~/.tmux.conf`:

```bash
# ═══════════════════════════════════════════════
# tmux configuration — Tmux Mastery Guide
# ═══════════════════════════════════════════════

# ─── Prefix ───
set -g prefix C-a
unbind C-b
bind C-a send-prefix

# ─── Sensible Defaults ───
set -g base-index 1
setw -g pane-base-index 1
set -g renumber-windows on
set -g history-limit 50000
set -sg escape-time 0
set -g mouse on
set -g allow-rename off
set -g default-terminal "tmux-256color"
set -ag terminal-overrides ",xterm-256color:RGB"
set -g display-time 3000
set -g display-panes-time 3000
set -g focus-events on

# ─── Key Bindings: Splits ───
bind | split-window -h -c "#{pane_current_path}"
bind - split-window -v -c "#{pane_current_path}"
unbind %
unbind '"'
bind c new-window -c "#{pane_current_path}"

# ─── Key Bindings: Navigation ───
bind -n M-Left select-pane -L
bind -n M-Right select-pane -R
bind -n M-Up select-pane -U
bind -n M-Down select-pane -D
bind -n S-Left previous-window
bind -n S-Right next-window

# ─── Key Bindings: Resize ───
bind -r S-Left resize-pane -L 5
bind -r S-Right resize-pane -R 5
bind -r S-Up resize-pane -U 5
bind -r S-Down resize-pane -D 5

# ─── Key Bindings: Utility ───
bind r source-file ~/.tmux.conf \; display "Config reloaded!"
bind S setw synchronize-panes \; display "Sync #{?synchronize-panes,ON,OFF}"

# ─── Vi Mode ───
set -g mode-keys vi
set -g status-keys vi
bind -T copy-mode-vi v send -X begin-selection
bind -T copy-mode-vi y send -X copy-pipe-and-cancel "pbcopy"
# Linux: replace "pbcopy" with "xclip -selection clipboard"

# ─── Status Bar ───
set -g status-position bottom
set -g status-style 'bg=#1E1E1E,fg=#D4D4D4'
set -g status-left-length 30
set -g status-left '#[fg=#FF9900,bold] #S #[default]│ '
set -g status-right '#[fg=#687078]%H:%M  %d-%b-%Y '
set -g window-status-format '#[fg=#687078] #I:#W '
set -g window-status-current-format '#[fg=#527FFF,bold] #I:#W* '
set -g window-status-separator ''
set -g pane-border-style 'fg=#444444'
set -g pane-active-border-style 'fg=#527FFF'

# ─── Plugins ───
set -g @plugin 'tmux-plugins/tpm'
set -g @plugin 'tmux-plugins/tmux-sensible'
set -g @plugin 'tmux-plugins/tmux-resurrect'
set -g @plugin 'tmux-plugins/tmux-continuum'
set -g @plugin 'tmux-plugins/tmux-yank'

# Plugin settings
set -g @continuum-save-interval '15'
set -g @continuum-restore 'on'
set -g @resurrect-processes 'vim nvim ssh top htop'

# Initialize TPM (keep at the very bottom)
run '~/.tmux/plugins/tpm/tpm'
```

## 4.4 Quick Reference: tmux Command-Line Interface

These are commands you run from your shell (outside tmux):

```bash
# ─── Sessions ───
tmux                          # New unnamed session
tmux new -s name              # New named session
tmux new -As name             # Attach or create
tmux attach -t name           # Attach to session
tmux attach -dt name          # Attach and detach others
tmux ls                       # List sessions
tmux kill-session -t name     # Kill one session
tmux kill-server              # Kill everything

# ─── Windows (from shell) ───
tmux new-window -t session    # New window in session
tmux select-window -t sess:2  # Switch to window 2

# ─── Scripting ───
tmux send-keys -t target "cmd" C-m    # Type command in pane
tmux split-window -h -t target        # Split pane horizontally
tmux split-window -v -t target        # Split pane vertically
tmux select-layout -t target tiled    # Apply layout

# ─── Output Capture ───
tmux capture-pane -t target -p        # Print pane content
tmux capture-pane -t target -p -S -   # Print full scrollback
tmux pipe-pane -t target 'cat >> log' # Live-log to file
tmux pipe-pane -t target              # Stop logging

# ─── Info ───
tmux -V                       # Version
tmux info                     # Server info
tmux show -g                  # Show global options
tmux lsk -N                   # List keybindings with descriptions
```

### Target Syntax

```
session:window.pane

Examples:
  work:1.0        → session "work", window 1, pane 0
  work:code       → session "work", window named "code"
  :2              → current session, window 2
  work            → session "work", current window/pane
```

## 4.5 Recommended Resources

### Official

| Resource | URL |
|----------|-----|
| tmux GitHub | [github.com/tmux/tmux](https://github.com/tmux/tmux) |
| Getting Started Wiki | [github.com/tmux/tmux/wiki/Getting-Started](https://github.com/tmux/tmux/wiki/Getting-Started) |
| Man page | `man tmux` (run in terminal) |
| FAQ | [github.com/tmux/tmux/wiki/FAQ](https://github.com/tmux/tmux/wiki/FAQ) |

### Plugins

| Plugin | Purpose |
|--------|---------|
| [TPM](https://github.com/tmux-plugins/tpm) | Plugin manager |
| [tmux-resurrect](https://github.com/tmux-plugins/tmux-resurrect) | Save/restore sessions |
| [tmux-continuum](https://github.com/tmux-plugins/tmux-continuum) | Auto-save sessions |
| [tmux-yank](https://github.com/tmux-plugins/tmux-yank) | System clipboard |
| [tmux-sensible](https://github.com/tmux-plugins/tmux-sensible) | Universal defaults |

### AI Tool Integration

| Resource | URL |
|----------|-----|
| Claude Code + tmux workflow | [pedroalonso.net](https://www.pedroalonso.net/blog/remote-ai-dev-workflow-claude-code-tmux-4090) |
| dmux (agent multiplexer) | [github.com/standardagents/dmux](https://github.com/standardagents/dmux) |
| Using tmux with Claude Code | [hboon.com](https://hboon.com/using-tmux-with-claude-code/) |

### Alternatives Worth Knowing About

| Tool | When to consider |
|------|-----------------|
| [Zellij](https://zellij.dev/) | If you want zero-config, beginner-friendly multiplexing |
| [GNU Screen](https://www.gnu.org/software/screen/) | If it's already installed and you just need basic session persistence |

---

## 🎉 Notebook Complete!

You now have a comprehensive understanding of tmux — from the fundamentals through hands-on practice to a production-ready configuration.

**Your learning path from here:**

1. **This week:** Use tmux for ALL terminal work. Refer to the cheat sheet (section 4.1) when you forget a keybinding.
2. **Week 2:** Customize your `.tmux.conf` further — change colors, add keybindings that match your workflow.
3. **Week 3:** Write a startup script for your most common project layout.
4. **Ongoing:** Explore plugins as needs arise. The tmux ecosystem is vast.

> **The single most important habit:** Always start your terminal work with `tmux new -As main`. After a week, it'll be automatic. After a month, you won't remember how you worked without it.

Study well! 📚